# 30-Day Hospital Readmission Prediction with Social Determinants of Health (MIMIC-IV)

This notebook builds feature matrices from MIMIC-IV (demographics, diagnoses, SDOH Z-codes, and NLP features from discharge notes), then trains Random Forest models to test whether SDOH improves readmission prediction.

**Data:** MIMIC-IV is a credentialed-access dataset on PhysioNet. No data or outputs are included in this repository. To run it, place `admissions.csv.gz`, `diagnoses_icd.csv.gz`, `patients.csv.gz` (MIMIC-IV hosp module) and `discharge.csv.zip` (MIMIC-IV-Note) in the working directory.

This is a research working notebook, so it follows the order the work was done in rather than a polished pipeline.

In [ ]:
import pandas as pd

mimic_admissions = pd.read_csv('admissions.csv.gz', compression='gzip')
mimic_diagnoses = pd.read_csv('diagnoses_icd.csv.gz', compression='gzip')
mimic_patients = pd.read_csv('patients.csv.gz', compression='gzip')

print(mimic_admissions.shape, mimic_diagnoses.shape, mimic_patients.shape)

In [ ]:
demo_cols = ['insurance', 'race', 'language', 'marital_status']

missing_demo = (mimic_admissions[demo_cols].isnull().sum() / len(mimic_admissions) * 100).round(2)

print("Missing % for each demographic column (out of 546,028 visits):")
print(missing_demo)

print("\nColumns exceeding 5% missing:")
print(missing_demo[missing_demo > 5])

In [ ]:
M0 = mimic_admissions[['subject_id', 'hadm_id', 'insurance', 'race', 'language', 'marital_status']].copy()
M0[['insurance', 'race', 'language', 'marital_status']] = M0[['insurance', 'race', 'language', 'marital_status']].fillna('Unknown')

M0 = M0.merge(mimic_patients[['subject_id', 'anchor_age']], on='subject_id', how='left')

print("M0 shape:", M0.shape)
print(M0.head())

M0.to_csv('M0_baseline.csv', index=False)
print("\nSaved M0_baseline.csv")

Main Table / Baseline Dataset (M0)
-----


What is the main table?
mimic_admissions is our main table. It has 546,028 rows — one row per hospital visit (hadm_id) — and includes subject_id, admission/discharge times, admission type, and 4 demographic columns (insurance, race, language, marital_status).

Why mimic_admissions is the main table:
---------


1)It's the only table structured at the visit level, matching our unit of analysis (we predict outcomes per hospital visit, not per patient).


2)Every other data source connects to it — diagnoses and notes link via hadm_id; patient demographics and Z-codes link via subject_id. Both keys already exist in this table, making it the natural hub.


3)It mirrors the structure used in Hernandez, Tchoua et al. (2024), where the main table was the referral record — one row per instance, with additional data merged onto it afterward.


4)Our target variable (30-day readmission) is calculated directly from this table's admittime and dischtime columns.

M0, N0, D0 — what they mean:
--------

M0 is the name of our baseline dataset — the simplest version of our data, with no diagnosis and no SDOH.


N0 is the number of rows (encounters/visits) in M0.


D0 is the number of features (columns) in M0.

M0 results:
-------

N0 = 546,028 encounters

D0 = 5 features: insurance, race, language, marital_status, anchor_age

(7 columns total = 2 ID columns + 5 features. anchor_age is a patient-level trait, so it repeats across all of a patient's visits.)

In [ ]:
# look at most common diagnoses (excluding Z-codes, which are SDOH not medical)
medical_diagnoses = mimic_diagnoses[
    ~mimic_diagnoses['icd_code'].astype(str).str.startswith(('Z55','Z56','Z57','Z58','Z59','Z60','Z61','Z62','Z63','Z64','Z65'))
]

top_diagnoses = medical_diagnoses['icd_code'].value_counts().head(30)
print(top_diagnoses)

This shows us the 30 most common medical diagnosis codes across all admissions (excluding the SDOH Z-codes, which we've already handled separately). we'll use this to figure out which diagnoses are common enough to hit that 80% completeness bar.

In [ ]:
total_admissions = len(mimic_admissions)

top_diagnoses_pct = (medical_diagnoses['icd_code'].value_counts().head(30) / total_admissions * 100).round(2)
print(f"Total admissions: {total_admissions}\n")
print(top_diagnoses_pct)

Diagnosis Coverage Check
----

We checked what % of all 546,028 admissions have each of the top 30 diagnosis codes. Even the most common one (hypertension, code 4019) only covers 18.75% — nowhere near the 80% threshold from our meeting.

Why: No single diagnosis is common enough across a diverse patient population to hit 80% on its own — same sparsity problem we saw with Z-codes.

Next step:
-----

Group related codes together (e.g., combine 4019 and I10, which are both hypertension) using the first 3 characters of each ICD code. This should raise coverage since multiple codes for the same condition get merged into one feature.

In [ ]:
# group diagnoses by first 3 characters of ICD code (broad diagnosis category)
diag_grouped = mimic_diagnoses[['hadm_id', 'icd_code']].copy()
diag_grouped['icd_group'] = diag_grouped['icd_code'].astype(str).str[:3]

# how many unique visits have each diagnosis GROUP (not individual code)
total_admissions = len(mimic_admissions)
group_coverage = diag_grouped.groupby('icd_group')['hadm_id'].nunique().sort_values(ascending=False)
group_coverage_pct = (group_coverage / total_admissions * 100).round(2)

print("Top 30 diagnosis groups by coverage:")
print(group_coverage_pct.head(30))

Grouping helped only a little. The largest diagnosis group increased from 18.75% to 19.47%, which is still far from the 80% target.

Largest group: Hypertension-related (401) – 19.47%
All other groups: Between 7% and 17%

This shows that an 80% threshold is not realistic. In a large hospital dataset, patients are admitted for many different conditions, so no single diagnosis or diagnosis group covers most patients. Grouping similar diagnoses cannot solve this because the population is naturally diverse.

In [ ]:
# exclude non-medical codes: Z-codes (history/status), V-codes (admin/procedure), Y-codes (metadata)
medical_diagnoses_clean = mimic_diagnoses[
    ~mimic_diagnoses['icd_code'].astype(str).str.startswith(('Z', 'V', 'Y'))
]

print("Rows before filtering:", len(mimic_diagnoses))
print("Rows after filtering:", len(medical_diagnoses_clean))
print("Rows removed:", len(mimic_diagnoses) - len(medical_diagnoses_clean))

# group by first 3 characters of ICD code
diag_grouped_clean = medical_diagnoses_clean[['hadm_id', 'icd_code']].copy()
diag_grouped_clean['icd_group'] = diag_grouped_clean['icd_code'].astype(str).str[:3]

# coverage % per group
total_admissions = len(mimic_admissions)
group_coverage_clean = diag_grouped_clean.groupby('icd_group')['hadm_id'].nunique().sort_values(ascending=False)
group_coverage_clean_pct = (group_coverage_clean / total_admissions * 100).round(2)

print("\nTop 30 medical diagnosis groups by coverage (Z/V/Y excluded):")
print(group_coverage_clean_pct.head(30))

Filtering Out Non-Medical Diagnosis Codes
-------
Our earlier diagnosis list included several codes that aren't actual medical diagnoses — administrative history flags (Z-codes like "history of tobacco use"), procedure/encounter codes (V-codes like "aftercare"), and metadata codes (Y-codes like "place where injury occurred"). These don't represent a patient's current medical condition, so we removed them before building M1.

We filtered out any ICD code starting with Z, V, or Y, keeping only genuine active medical diagnoses. We then re-grouped the remaining codes by their first 3 characters (combining related codes like hypertension's ICD-9 and ICD-10 versions into one category) and recalculated what percentage of all 546,028 admissions each diagnosis group covers.

In [ ]:
# apply 10% threshold to cleaned diagnosis groups
threshold_pct = 0.10
threshold_count = total_admissions * threshold_pct

kept_groups = group_coverage_clean[group_coverage_clean >= threshold_count].index.tolist()
print(f"10% threshold ({threshold_count:.0f} visits minimum): {len(kept_groups)} diagnosis groups survive")
print(kept_groups)

# FILTER FIRST — only keep rows belonging to the 8 surviving groups
diag_grouped_small = diag_grouped_clean[diag_grouped_clean['icd_group'].isin(kept_groups)]

# now crosstab only sees 8 groups, not 700+
diag_dummies = pd.crosstab(diag_grouped_small['hadm_id'], diag_grouped_small['icd_group'])
diag_dummies = (diag_dummies > 0).astype(int)
diag_dummies = diag_dummies.reset_index()

print("\nDiagnosis dummies shape:", diag_dummies.shape)
print(diag_dummies.head())

8 diagnosis groups survived the 10% threshold:
-----

250 — diabetes mellitus

272 — lipid metabolism disorders (high cholesterol)

401 — essential hypertension

E11 — type 2 diabetes

E78 — lipid metabolism disorders (ICD-10 version)

E87 — fluid/electrolyte imbalance

I10 — essential hypertension (ICD-10 version)

K21 — GERD/acid reflux

In [ ]:
kept_groups_cols = ['250', '272', '401', 'E11', 'E78', 'E87', 'I10', 'K21']

M1 = M0.merge(diag_dummies, on='hadm_id', how='left')
M1[kept_groups_cols] = M1[kept_groups_cols].fillna(0).astype(int)

print("M1 shape:", M1.shape)
print(M1.head())

M1.to_csv('M1_ehr.csv', index=False)
print("\nSaved M1_ehr.csv")

M1 results:
-------

N1 = 546,028 encounters

D1 = 13 features: insurance, race, language, marital_status, anchor_age,

 250 (diabetes), 272 (high cholesterol), 401 (hypertension),
     E11 (type 2 diabetes), E78 (high cholesterol), E87 (fluid/electrolyte imbalance),
     I10 (hypertension), K21 (GERD/acid reflux)

(15 columns total = 2 ID columns + 5 baseline features + 8 diagnosis features.


N1 stayed the same as N0 since visits with none of these 8 diagnoses were kept
and filled with 0, rather than dropped. 345,317 of the 546,028 visits (63.2%)
have at least one of these 8 diagnoses.)

Diagnosis groups were selected using a 10% coverage threshold (at least 54,603
of 546,028 visits must have that diagnosis) — the same 80% rule from the meeting
was not achievable, since even the most common individual diagnosis only reaches
18.75% coverage. Non-medical Z/V/Y administrative codes were excluded before
grouping, and related ICD-9/ICD-10 codes were merged (e.g., 401 and I10 both
represent hypertension).

In [ ]:
# rebuild Z-code SDOH extraction
sdoh_codes = mimic_diagnoses[
    (mimic_diagnoses['icd_version'] == 10) &
    (mimic_diagnoses['icd_code'].str.startswith(('Z55', 'Z56', 'Z57', 'Z58', 'Z59', 'Z60', 'Z61', 'Z62', 'Z63', 'Z64', 'Z65')))
].copy()

print("SDOH rows:", len(sdoh_codes))

# one-hot encode per patient
sdoh_dummies = sdoh_codes[['subject_id', 'icd_code']].copy()
sdoh_dummies['value'] = 1
sdoh_wide = sdoh_dummies.pivot_table(index='subject_id', columns='icd_code', values='value', aggfunc='max', fill_value=0).reset_index()

print("sdoh_wide shape:", sdoh_wide.shape)
sdoh_wide.to_csv('sdoh_dummies.csv', index=False)
print("Saved sdoh_dummies.csv")

This gives us 6,949 SDOH diagnosis records across 3,749 unique patients, spread across 74 distinct Z-codes.

In [ ]:
zcode_threshold_pct = 0.05  # 5% experiment first
total_zcode_patients = len(sdoh_wide)
zcode_threshold_count = total_zcode_patients * zcode_threshold_pct

zcode_counts = sdoh_wide.drop(columns=['subject_id']).sum().sort_values(ascending=False)
kept_zcodes = zcode_counts[zcode_counts >= zcode_threshold_count].index.tolist()

print(f"5% threshold ({zcode_threshold_count:.0f} patients minimum): {len(kept_zcodes)} columns survive")
print(kept_zcodes)

sdoh_filtered = sdoh_wide[['subject_id'] + kept_zcodes]

M2 = M0.merge(sdoh_filtered, on='subject_id', how='left')
M2[kept_zcodes] = M2[kept_zcodes].fillna(0).astype(int)

print("\nM2 shape:", M2.shape)
print(M2.head())

M2.to_csv('M2_sdoh_5pct.csv', index=False)
print("\nSaved M2_sdoh_5pct.csv")

applying 5% threshold on z-code
----
columns survive out of 75:

Z-code ,	 Description ,	Coverage

Z590	, Homelessness ,	58.02%

Z62810 ,	Physical abuse in childhood ,	9.23%

Z560 ,	Unemployment ,	7.50%

Z5900 ,	Homelessness unspecified ,	6.75%

Z602 ,	Problems related to living alone ,	6.08%



In [ ]:
zcode_threshold_pct_10 = 0.10
zcode_threshold_count_10 = total_zcode_patients * zcode_threshold_pct_10

kept_zcodes_10 = zcode_counts[zcode_counts >= zcode_threshold_count_10].index.tolist()
print(f"10% threshold ({zcode_threshold_count_10:.0f} patients minimum): {len(kept_zcodes_10)} columns survive")
print(kept_zcodes_10)

sdoh_filtered_10 = sdoh_wide[['subject_id'] + kept_zcodes_10]

M2_10 = M0.merge(sdoh_filtered_10, on='subject_id', how='left')
M2_10[kept_zcodes_10] = M2_10[kept_zcodes_10].fillna(0).astype(int)

print("\nM2 (10%) shape:", M2_10.shape)
print(M2_10.head())

M2_10.to_csv('M2_sdoh_10pct.csv', index=False)
print("\nSaved M2_sdoh_10pct.csv")

applying 10% threshold on z-code
------

columns survive out of 75:

Z-code , Description , Coverage

Z590 , Homelessness , 58.02%

In [ ]:
import zipfile

with zipfile.ZipFile('discharge.csv.zip', 'r') as z:
    print(z.namelist())
    z.extract('discharge.csv')

discharge = pd.read_csv('discharge.csv', nrows=1000)
print("Shape:", discharge.shape)
print("Columns:", discharge.columns.tolist())

In [ ]:
sdoh_keywords = {
    'homeless': ['homeless', 'homelessness', 'shelter', 'unhoused', 'no fixed address'],
    'alcohol_use': ['alcohol', 'drinking', 'ethanol', 'etoh', 'drinks per day'],
    'drug_use': ['drug use', 'substance use', 'cocaine', 'heroin', 'illicit drug'],
    'substance_abuse': ['substance abuse', 'alcohol abuse', 'drug abuse',
                        'history of abuse', 'abuse of alcohol', 'abuse of drugs'],
    'tobacco_use': ['smoking', 'tobacco', 'cigarette', 'smoker', 'quit smoking'],
    'lives_alone': ['lives alone', 'living alone', 'alone at home'],
    'unemployed': ['unemployed', 'unemployment', 'out of work', 'not working'],
    'food_insecure': ['food insecure', 'food insecurity', 'hungry', 'not enough food'],
    'domestic_violence': ['domestic violence', 'intimate partner violence',
                          'physically abused by partner', 'battered',
                          'unsafe at home', 'abused by spouse'],
    'limited_support': ['no family support', 'no social support', 'isolated', 'no support'],
    'financial_stress': ['financial stress', 'financial difficulty', "can't afford", 'cannot afford'],
}

def extract_sdoh(text):
    text_lower = str(text).lower()
    return {category: (1 if any(kw in text_lower for kw in keywords) else 0)
            for category, keywords in sdoh_keywords.items()}

print(extract_sdoh(discharge['text'][0]))

In [ ]:
chunk_size = 10000
results = []

for chunk in pd.read_csv('discharge.csv', chunksize=chunk_size):
    sdoh_results = chunk['text'].apply(extract_sdoh)
    chunk_extracted = pd.concat([chunk[['subject_id', 'hadm_id']],
                                  pd.DataFrame(sdoh_results.tolist())], axis=1)
    results.append(chunk_extracted)

sdoh_nlp_full = pd.concat(results, ignore_index=True)
print("Full NLP dataset shape:", sdoh_nlp_full.shape)
print(sdoh_nlp_full.drop(columns=['subject_id', 'hadm_id']).sum())

sdoh_nlp_full.to_csv('sdoh_nlp_full.csv', index=False)
print("Saved sdoh_nlp_full.csv")

Percentage of all 653,586 notes
------

alcohol_use: 108,323 → 16.57%

tobacco_use: 43,933 → 6.72%

drug_use: 27,851 → 4.26%

substance_abuse: 22,834 → 3.49%

limited_support: 20,670 → 3.16%

homeless: 4,048 → 0.62%

unemployed: 3,507 → 0.54%

food_insecure: 3,396 → 0.52%

lives_alone: 2,918 → 0.45%

domestic_violence: 1,135 → 0.17%

financial_stress: 730 → 0.11%

In [ ]:
nlp_for_merge = sdoh_nlp_full.drop(columns=['subject_id'])
nlp_cols = nlp_for_merge.columns.drop('hadm_id')

M2_full_5pct = M2.merge(nlp_for_merge, on='hadm_id', how='left')
M2_full_5pct[nlp_cols] = M2_full_5pct[nlp_cols].fillna(0).astype(int)

print("M2_full_5pct shape:", M2_full_5pct.shape)
print(M2_full_5pct.head())

M2_full_5pct.to_csv('M2_full_5pct.csv', index=False)
print("Saved M2_full_5pct.csv")

That worked correctly — 546,028 rows (same visit count as always, nothing lost or duplicated) and

23 columns total:

subject_id, hadm_id,

5 baseline features (insurance, race, language, marital_status, anchor_age),

5 surviving Z-code columns from the 5% threshold,
  
and 11 NLP columns.

In [ ]:
M2_full_10pct = M2_10.merge(nlp_for_merge, on='hadm_id', how='left')
M2_full_10pct[nlp_cols] = M2_full_10pct[nlp_cols].fillna(0).astype(int)

print("M2_full_10pct shape:", M2_full_10pct.shape)
print(M2_full_10pct.head())

M2_full_10pct.to_csv('M2_full_10pct.csv', index=False)
print("Saved M2_full_10pct.csv")

on using 10% threshhold
----

the column comes down to 19; 1 surviving zcode

In [ ]:
# check missing % per column for both M2 versions
missing_5pct = (M2_full_5pct.isnull().sum() / len(M2_full_5pct) * 100).round(2)
missing_10pct = (M2_full_10pct.isnull().sum() / len(M2_full_10pct) * 100).round(2)

print("M2_full_5pct missing %:\n", missing_5pct[missing_5pct > 0])
print("\nM2_full_10pct missing %:\n", missing_10pct[missing_10pct > 0])

In [ ]:
missing_M0 = (M0.isnull().sum() / len(M0) * 100).round(2)
missing_M1 = (M1.isnull().sum() / len(M1) * 100).round(2)

print("M0 missing %:\n", missing_M0[missing_M0 > 0])
print("\nM1 missing %:\n", missing_M1[missing_M1 > 0])

In [ ]:
readmit_check = mimic_admissions[['subject_id', 'hadm_id', 'admittime', 'dischtime', 'admission_type']].copy()
readmit_check['admittime'] = pd.to_datetime(readmit_check['admittime'])
readmit_check['dischtime'] = pd.to_datetime(readmit_check['dischtime'])

# sort by patient and admission time
readmit_check = readmit_check.sort_values(['subject_id', 'admittime']).reset_index(drop=True)

print(readmit_check.shape)
print(readmit_check.head(10))

In [ ]:
readmit_check['next_admittime'] = readmit_check.groupby('subject_id')['admittime'].shift(-1)
readmit_check['next_admission_type'] = readmit_check.groupby('subject_id')['admission_type'].shift(-1)

readmit_check['days_to_next_admit'] = (readmit_check['next_admittime'] - readmit_check['dischtime']).dt.days

print(readmit_check[['subject_id', 'hadm_id', 'dischtime', 'next_admittime', 'days_to_next_admit']].head(10))

In [ ]:
readmit_check['readmit_30_any'] = (readmit_check['days_to_next_admit'] <= 30).astype(int)
readmit_check['readmit_30_same_type'] = (
    (readmit_check['days_to_next_admit'] <= 30) &
    (readmit_check['admission_type'] == readmit_check['next_admission_type'])
).astype(int)

print(readmit_check[['subject_id', 'hadm_id', 'days_to_next_admit', 'admission_type', 'next_admission_type', 'readmit_30_any', 'readmit_30_same_type']].head(10))

print("\n=== Any-cause readmission rate ===")
print(readmit_check['readmit_30_any'].value_counts())

print("\n=== Same-type readmission rate ===")
print(readmit_check['readmit_30_same_type'].value_counts())

In [ ]:
# Re-run extraction on a sample, then pull a mix of flagged/unflagged notes for review
sample = discharge.sample(n=200, random_state=42).copy()
sample_results = sample['text'].apply(extract_sdoh)
sample_flags = pd.DataFrame(sample_results.tolist())

sample_combined = pd.concat([sample[['note_id', 'text']].reset_index(drop=True), sample_flags], axis=1)

# pick 5 notes flagged for each category, plus 10 with no flags at all
review_rows = []
for category in sdoh_keywords.keys():
    flagged = sample_combined[sample_combined[category] == 1].head(5)
    review_rows.append(flagged)

no_flags = sample_combined[sample_combined[list(sdoh_keywords.keys())].sum(axis=1) == 0].head(10)
review_rows.append(no_flags)

review_sample = pd.concat(review_rows).drop_duplicates(subset='note_id')
review_sample.to_csv('nlp_review_sample.csv', index=False)
print("Saved nlp_review_sample.csv, shape:", review_sample.shape)

In [ ]:
total_patients = mimic_patients['subject_id'].nunique()  # full patient population as the denominator
print("Total patients in dataset:", total_patients)

col_counts = sdoh_wide.drop(columns=['subject_id']).sum().sort_values(ascending=False)
col_pct_of_total = (col_counts / total_patients * 100).round(4)

zcode_coverage_table = pd.DataFrame({
    'patients_with_code': col_counts,
    'pct_of_total_patients': col_pct_of_total
})

print(zcode_coverage_table.to_string())

The Core Difference
--------


Earlier, we calculated percentages using only the 3,749 patients who had at least one Z-code. In that group, homelessness looked common (58%).

Now, we calculated percentages using all 364,627 patients in MIMIC-IV. With this larger population, homelessness drops to only 0.60% because most patients have no Z-codes.

Why this matters: Professor Tchoua wanted percentages based on the entire patient population to show how common each SDOH indicator really is. These results show that all Z-codes are very rare.

Because of this, a 5% or 10% threshold would remove every Z-code column, since none of them even reach 1% of the full patient population. This means a percentage-based threshold is not suitable for this dataset.

In [ ]:
zcode_coverage_table.to_csv('zcode_coverage_full_population.csv')
print("Saved zcode_coverage_full_population.csv")

In [ ]:
# patients who have at least one Z-code documented at all
complete_sdoh_patients = sdoh_wide['subject_id'].unique()
print("Patients with at least one SDOH code:", len(complete_sdoh_patients))

M2_complete = M0[M0['subject_id'].isin(complete_sdoh_patients)].merge(sdoh_wide, on='subject_id', how='left')
M2_complete = M2_complete.fillna(0)

print("M2_complete shape:", M2_complete.shape)

24,330 Rows and 81 Columns
-----

The 24,330 rows represent hospital visits, not patients. These visits belong to the 3,749 patients who have at least one Z-code. Since one patient can have multiple hospital visits(on avg 6.5 visit per patient), these patients account for 24,330 total admissions.

The 81 columns include:
-------

2 ID columns: subject_id and hadm_id

5 baseline features: insurance, race, language, marital status, and anchor age

74 Z-code columns, with no threshold filtering applied yet.

In [ ]:
# 5% threshold, within the 3,749 SDOH-coded patients
threshold_pct = 0.05
threshold_count = 3749 * threshold_pct

zcode_counts_cohort = sdoh_wide.drop(columns=['subject_id']).sum().sort_values(ascending=False)
zcode_pct_cohort = (zcode_counts_cohort / 3749 * 100).round(2)

kept_zcodes_5pct = zcode_counts_cohort[zcode_counts_cohort >= threshold_count].index.tolist()

print(f"5% threshold ({threshold_count:.0f} of 3,749 patients minimum): {len(kept_zcodes_5pct)} columns survive")
print(kept_zcodes_5pct)

# 10% threshold, same cohort
threshold_count_10 = 3749 * 0.10
kept_zcodes_10pct = zcode_counts_cohort[zcode_counts_cohort >= threshold_count_10].index.tolist()

print(f"\n10% threshold ({threshold_count_10:.0f} of 3,749 patients minimum): {len(kept_zcodes_10pct)} columns survive")
print(kept_zcodes_10pct)

applying 5% threshold on z-code
----
columns survive out of 75:

Z-code ,	 Description ,	Coverage

Z590	, Homelessness ,	58.02%

Z62810 ,	Physical abuse in childhood ,	9.23%

Z560 ,	Unemployment ,	7.50%

Z5900 ,	Homelessness unspecified ,	6.75%

Z602 ,	Problems related to living alone ,	6.08%



applying 10% threshold on z-code
----
columns survive out of 75:

Z-code ,	 Description ,	Coverage

Z590	, Homelessness ,	58.02%



In [ ]:
# "drop if more than 5% missing" = keep only if presence >= 95%
threshold_95 = 3749 * 0.95
kept_95 = zcode_counts_cohort[zcode_counts_cohort >= threshold_95].index.tolist()
print(f"Columns with <=5% missing (>=95% presence, >={threshold_95:.0f} patients): {len(kept_95)}")
print(kept_95)

# "drop if more than 10% missing" = keep only if presence >= 90%
threshold_90 = 3749 * 0.90
kept_90 = zcode_counts_cohort[zcode_counts_cohort >= threshold_90].index.tolist()
print(f"\nColumns with <=10% missing (>=90% presence, >={threshold_90:.0f} patients): {len(kept_90)}")
print(kept_90)

**No column survives a "drop if more than 5% missing" rule.**
----

In [ ]:
readmit_cols = ['hadm_id', 'readmit_30_any', 'readmit_30_same_type']

# M0
M0 = M0.merge(readmit_check[readmit_cols], on='hadm_id', how='left')
M0[['readmit_30_any', 'readmit_30_same_type']] = M0[['readmit_30_any', 'readmit_30_same_type']].fillna(0).astype(int)

# M1
M1 = M1.merge(readmit_check[readmit_cols], on='hadm_id', how='left')
M1[['readmit_30_any', 'readmit_30_same_type']] = M1[['readmit_30_any', 'readmit_30_same_type']].fillna(0).astype(int)

print("M0 shape:", M0.shape)
print("M1 shape:", M1.shape)
print(M0[['subject_id', 'hadm_id', 'readmit_30_any', 'readmit_30_same_type']].head())


We merged the two readmission label columns (readmit_30_any, readmit_30_same_type) from readmit_check onto M0 and M1, joining on hadm_id.

M0 shape: (546028, 9) — M0 now has 9 columns (was 7: subject_id, hadm_id, insurance, race, language, marital_status, anchor_age → +2 label columns), still 546,028 rows, so nothing was lost.


M1 shape: (546028, 17) — M1 now has 17 columns (was 15: 2 IDs + 5 baseline + 8 diagnosis flags → +2 label columns), also still 546,028 rows.

In [ ]:
# M2_full_5pct
M2_full_5pct = M2_full_5pct.merge(readmit_check[readmit_cols], on='hadm_id', how='left')
M2_full_5pct[['readmit_30_any', 'readmit_30_same_type']] = M2_full_5pct[['readmit_30_any', 'readmit_30_same_type']].fillna(0).astype(int)

# M2_full_10pct
M2_full_10pct = M2_full_10pct.merge(readmit_check[readmit_cols], on='hadm_id', how='left')
M2_full_10pct[['readmit_30_any', 'readmit_30_same_type']] = M2_full_10pct[['readmit_30_any', 'readmit_30_same_type']].fillna(0).astype(int)

print("M2_full_5pct shape:", M2_full_5pct.shape)
print("M2_full_10pct shape:", M2_full_10pct.shape)

M2_full_5pct.to_csv('M2_full_5pct.csv', index=False)
M2_full_10pct.to_csv('M2_full_10pct.csv', index=False)
print("Re-saved both M2 CSVs with readmission labels included")

This confirms the readmission labels are now attached to both M2 versions.
-------

M2_full_5pct: (546028, 25) — was 23 columns before (2 IDs + 5 baseline + 5 Z-codes + 11 NLP), now 25 after adding the 2 readmission label columns (readmit_30_any, readmit_30_same_type). Row count unchanged at 546,028 — no visits lost.

M2_full_10pct: (546028, 21) — was 19 columns before (2 IDs + 5 baseline + 1 Z-code + 11 NLP), now 21 after the same 2 label columns added. Also still 546,028 rows.

In [ ]:
def summarize(name, df):
    n = df.shape[0]
    d = df.shape[1] - 2 - 2  # minus 2 ID cols, minus 2 label cols
    print(f"{name}: N={n:,} encounters, D={d} features")

summarize("M0 (baseline)", M0)
summarize("M1 (EHR)", M1)
summarize("M2 (5% Z-code)", M2_full_5pct)
summarize("M2 (10% Z-code)", M2_full_10pct)


In [ ]:
import matplotlib.pyplot as plt

elbow_data = group_coverage_clean_pct.sort_values(ascending=False)

plt.figure(figsize=(10, 6))
plt.plot(range(1, len(elbow_data) + 1), elbow_data.values, marker='o')
plt.xlabel("Number of diagnosis groups kept (ranked by coverage)")
plt.ylabel("% of visits covered")
plt.title("Elbow Plot: Diagnosis Group Coverage")
plt.grid(True, alpha=0.3)
plt.axhline(y=10, color='red', linestyle='--', label='10% threshold')
plt.legend()
plt.show()

In [ ]:
elbow_data = group_coverage_clean_pct.sort_values(ascending=False).head(30)

In [ ]:
import matplotlib.pyplot as plt

elbow_data = group_coverage_clean_pct.sort_values(ascending=False).head(30)

plt.figure(figsize=(10, 6))
plt.plot(range(1, len(elbow_data) + 1), elbow_data.values, marker='o')
plt.xlabel("Number of diagnosis groups kept (ranked by coverage)")
plt.ylabel("% of visits covered")
plt.title("Elbow Plot: Diagnosis Group Coverage")
plt.grid(True, alpha=0.3)
plt.axhline(y=10, color='red', linestyle='--', label='10% threshold')
plt.legend()
plt.show()

In [ ]:
print("Top 30 diagnosis groups:")
print(group_coverage_clean_pct.sort_values(ascending=False).head(30))

print("\nTop 15 diagnosis groups:")
print(group_coverage_clean_pct.sort_values(ascending=False).head(15))

Elbow Plot: Diagnosis Coverage
----

We plotted the top 30 diagnosis groups by how many visits they cover, to
show visually where a good cutoff point is.

There's a clear bend at 8 groups — coverage drops fast from 19.47% down to
about 10%, then flattens out right around there.

There's a smaller, softer bend around 15 groups too, before it keeps
slowly dropping after that.

So we have two reasonable choices: 8 groups (the sharpest bend, matches
our original 10% cutoff) or 15 groups (looser, keeps more diagnoses).

Building a New M1 (15 Diagnosis Groups)
-----

In [ ]:
top15_groups = group_coverage_clean_pct.sort_values(ascending=False).head(15)
print(top15_groups)

In [ ]:
top15_groups = group_coverage_clean_pct.sort_values(ascending=False).head(15)
kept_groups_15 = top15_groups.index.tolist()

diag_dummies_15 = diag_grouped_clean[diag_grouped_clean['icd_group'].isin(kept_groups_15)]
diag_dummies_15 = pd.crosstab(diag_dummies_15['hadm_id'], diag_dummies_15['icd_group'])
diag_dummies_15 = (diag_dummies_15 > 0).astype(int).reset_index()

print(kept_groups_15)

In [ ]:
# rebuild from a clean M0-style baseline (no readmission labels yet)
M0_clean = mimic_admissions[['subject_id', 'hadm_id', 'insurance', 'race', 'language', 'marital_status']].copy()
M0_clean[['insurance', 'race', 'language', 'marital_status']] = M0_clean[['insurance', 'race', 'language', 'marital_status']].fillna('Unknown')
M0_clean = M0_clean.merge(mimic_patients[['subject_id', 'anchor_age']], on='subject_id', how='left')

M1_15 = M0_clean.merge(diag_dummies_15, on='hadm_id', how='left')
M1_15[kept_groups_15] = M1_15[kept_groups_15].fillna(0).astype(int)

print("M1_15 shape:", M1_15.shape)
print(M1_15.head())

M1_15.to_csv('M1_15diagnoses.csv', index=False)
print("Saved M1_15diagnoses.csv")

M1_15 Results:
----

N1 = 546,028 hospital visits (same as M0)

D1 = 20 features:

5 baseline features: insurance, race, language, marital_status, anchor_age

15 diagnosis features: 250 (diabetes), 272 (high cholesterol), 401 (hypertension), 276 (fluid/electrolyte imbalance), 285 (anemia), E11 (type 2 diabetes), E78 (high cholesterol), E87 (fluid/electrolyte imbalance), F32 (major depressive disorder), I10 (hypertension), I25 (chronic ischemic heart disease), K21 (GERD/acid reflux), 414 (chronic ischemic heart disease), 427 (cardiac dysrhythmias), 530 (esophagus disorders)

Total columns = 22 (2 ID columns + 5 baseline features + 15 diagnosis features).

The number of visits stayed the same because visits without any of these 15 diagnoses were kept and their diagnosis values were filled with 0 instead of being removed.

The 15 diagnosis groups were chosen using an elbow plot of the top 30 diagnosis groups. The plot showed a clear bend at 8 groups and another smaller bend at 15 groups. We used the 15-group cutoff to include more diagnosis information while still keeping the most common conditions. Non-medical Z/V/Y codes were removed before grouping, and matching ICD-9 and ICD-10 codes (such as 401/I10 for hypertension and 272/E78 for high cholesterol) were combined.----





In [ ]:
visits_with_diagnosis = (M1_15[kept_groups_15].sum(axis=1) > 0).sum()
pct_with_diagnosis = visits_with_diagnosis / len(M1_15) * 100

print(f"Visits with at least one of the 15 diagnoses: {visits_with_diagnosis:,} ({pct_with_diagnosis:.1f}%)")

M1_15: Expanding to 15 Diagnosis Groups
----

Compared to the original 8-diagnosis M1, using an elbow-justified cutoff of
15 diagnosis groups increases coverage: 400,075 of 546,028 visits (73.3%)
now have at least one of the 15 diagnoses, up from 345,317 visits (63.2%)
with the 8-diagnosis version.

Tradeoff: casting a wider net picks up more visits with real diagnostic
signal, but the additional 7 diagnosis groups (276, 285, F32, I25, 414,
427, 530) each have lower individual coverage (8.6%-9.6%) than the
original 8, which ranged from 10.4% up to 19.47%.

Both M1 (8 diagnoses) and M1_15 (15 diagnoses) are kept as separate,
comparable matrices for the professor to choose between.

Get the full missingness/presence table for every SDOH (Z-code) column — not just the ones already surviving a threshold.
----------


In [ ]:
sdoh_presence_table = pd.DataFrame({
    'patients_with_code': zcode_counts_cohort,
    'pct_of_cohort_3749': zcode_pct_cohort,
    'pct_missing_of_cohort': 100 - zcode_pct_cohort
}).sort_values('pct_of_cohort_3749', ascending=False)

print(sdoh_presence_table)
sdoh_presence_table.to_csv('sdoh_full_presence_table.csv')
print("\nSaved sdoh_full_presence_table.csv")

In [ ]:
pd.set_option('display.max_rows', 100)
print(sdoh_presence_table.to_string())

Full SDOH Presence/Missingness Table
----

Professor Tchoua asked for the full picture across all 74 Z-codes, not
just the ones that already survived a 5%/10% cutoff.

This table shows, for every single Z-code, how many of the 3,749
SDOH-coded patients have it, what % of the cohort that is, and what %
of the cohort is missing it.

No threshold applied here — this is the full unfiltered list, so the
cutoff decision for our next matrix (M3) can be made by looking at the
whole distribution, not just the codes that already passed a filter.

Saved as sdoh_full_presence_table.csv.

Are the Discharge Notes Written by Doctors or Community Health Workers?
-----

In [ ]:
print(discharge['note_type'].value_counts())
print(discharge['note_type'].unique())

In [ ]:
for i in range(3):
    print(f"=== Note {i} ===")
    print(discharge['text'].iloc[i][:1500])
    print("\n\n")

We know these are doctor-written notes because every note is labeled DS (Discharge Summary) in MIMIC. Each note also includes the medical service and the attending doctor. The notes contain clinical sections such as History of Present Illness, lab results, medications, and procedures, which are typical of a doctor's documentation. We also confirmed this by reading real discharge summaries from the dataset ourselves.

Build a new, stricter SDOH matrix ("Full SDOH" / M3): patients kept only if they have 90–95% completeness across a small set (5–10) of SDOH columns (stricter than M2_complete, which only requires ≥1 Z-code).
------

In [ ]:
top5_zcodes = zcode_pct_cohort.sort_values(ascending=False).head(5).index.tolist()
top10_zcodes = zcode_pct_cohort.sort_values(ascending=False).head(10).index.tolist()

print("Top 5:", top5_zcodes)
print("Top 10:", top10_zcodes)

In [ ]:
def completeness_stats(zcode_list, label):
    sub = sdoh_wide[['subject_id'] + zcode_list].copy()
    sub['pct_present'] = sub[zcode_list].sum(axis=1) / len(zcode_list) * 100

    print(f"\n=== {label} ({len(zcode_list)} columns) ===")
    print(sub['pct_present'].value_counts().sort_index(ascending=False))

    for pct_cutoff in [90, 50, 20, 10]:
        n_patients = (sub['pct_present'] >= pct_cutoff).sum()
        print(f"Patients with >= {pct_cutoff}% of these columns present: {n_patients}")

    return sub

sub5 = completeness_stats(top5_zcodes, "Top 5 SDOH")
sub10 = completeness_stats(top10_zcodes, "Top 10 SDOH")

SDOH Completeness Check (Testing M3)
--------------

We tested whether patients usually have multiple SDOH issues recorded together by looking at the top 5 and top 10 most common SDOH codes.

Top 5 SDOH codes

0 patients had 90% or more of the 5 codes.
Only 15 patients had 50% or more.
Most patients (2,750) had only 1 out of 5 codes (20%).
741 patients had none of the top 5 codes because their only SDOH code was outside this list.

Top 10 SDOH codes

0 patients had 90% or more of the 10 codes.
Only 4 patients had 50% or more.
Most patients (2,963) had only 1 out of 10 codes (10%).
What this tells us

A 90–95% completeness requirement is not realistic for this dataset. Most patients have one SDOH code or none, not several at the same time. This is expected because doctors usually record one important social issue during a hospital visit rather than completing a full SDOH screening.

In [ ]:
top5_zcodes = zcode_pct_cohort.sort_values(ascending=False).head(5).index.tolist()

# M3: patients with at least 1 of the top 5 SDOH columns present
# (denser than M2_complete's "at least 1 of all 74")
m3_patients = sdoh_wide[sdoh_wide[top5_zcodes].sum(axis=1) >= 1]['subject_id'].unique()
print("M3 patients (>=1 of top 5 SDOH codes):", len(m3_patients))

M3 = M0_clean[M0_clean['subject_id'].isin(m3_patients)].merge(
    sdoh_wide[['subject_id'] + top5_zcodes], on='subject_id', how='left'
)
M3[top5_zcodes] = M3[top5_zcodes].fillna(0).astype(int)

print("M3 shape:", M3.shape)
print(M3.head())

M3.to_csv('M3_draft.csv', index=False)
print("Saved M3_draft.csv")

M3: Attempting the "Complete SDOH" Matrix

Professor Tchoua's goal: Create a smaller dataset where patients have 90–95% completeness across 5–10 SDOH columns.

Result: This was not possible with the data. No patients had 90% or higher completeness using either the top 5 or top 10 SDOH columns. Even using a 50% threshold, only 15 patients (top 5) or 4 patients (top 10) qualified. Most patients have only one SDOH code, not multiple codes recorded together.

M3_draft (fallback): Instead, we created a smaller dataset containing patients with at least one of the top 5 most common SDOH codes.

M3_draft results

3,008 patients
20,061 hospital visits
5 baseline features
5 SDOH columns: Z590, Z62810, Z560, Z5900, Z602

Comparison with M2_complete

M2_complete: 3,749 patients with at least one of all 74 SDOH codes.
M3_draft: 3,008 patients with at least one of the top 5 most common SDOH codes, giving a smaller and denser dataset.

Conclusion: M3_draft is the closest practical version of Professor Tchoua's idea, but a true 90–95% complete SDOH matrix cannot be created from this dataset because patients rarely have multiple SDOH codes documented together.

Elbow Method for SDOH Codes
-------

In [ ]:
import matplotlib.pyplot as plt

elbow_sdoh_full = zcode_pct_cohort.sort_values(ascending=False)

plt.figure(figsize=(10, 6))
plt.plot(range(1, len(elbow_sdoh_full) + 1), elbow_sdoh_full.values, marker='o', markersize=4)
plt.xlabel("Number of SDOH (Z-code) columns kept (ranked by coverage)")
plt.ylabel("% of SDOH-coded cohort covered (out of 3,749)")
plt.title(f"Elbow Plot: All {len(elbow_sdoh_full)} SDOH Z-Codes")
plt.grid(True, alpha=0.3)
plt.axhline(y=5, color='red', linestyle='--', label='5% threshold')
plt.axhline(y=10, color='orange', linestyle='--', label='10% threshold')
plt.legend()
plt.show()

In [ ]:
import matplotlib.pyplot as plt

elbow_sdoh = zcode_pct_cohort.sort_values(ascending=False).head(30)

plt.figure(figsize=(10, 6))
plt.plot(range(1, len(elbow_sdoh) + 1), elbow_sdoh.values, marker='o')
plt.xlabel("Number of SDOH (Z-code) columns kept (ranked by coverage)")
plt.ylabel("% of SDOH-coded cohort covered (out of 3,749)")
plt.title("Elbow Plot: SDOH Z-Code Coverage")
plt.grid(True, alpha=0.3)
plt.axhline(y=5, color='red', linestyle='--', label='5% threshold')
plt.axhline(y=10, color='orange', linestyle='--', label='10% threshold')
plt.legend()
plt.show()

this shows only the top 30 of 74 total SDOH Z-codes for
readability — codes 31-74 are all under ~0.3% coverage and would
flatten the chart without adding useful information.

Working on M2
------

M2_complete: Known Limitations
----

This matrix keeps all patients with at least 1 of the 74 SDOH Z-codes
documented (3,749 patients, 24,330 visits), but has two known
limitations flagged directly by Professor Tchoua:

1. NLP/discharge-note columns are NOT merged into this version — only
   structured Z-codes are included. This means SDOH signal from the
   discharge notes (alcohol_use, homelessness mentions, etc.) is
   missing entirely here.

2. Even among these 3,749 "SDOH-coded" patients, most have only 1
   documented Z-code (see SDOH completeness check) — so many of the
   74 columns are still mostly zeros even within this filtered group.

Kept as-is for reference, but not recommended as the primary SDOH
matrix for modeling. M3_draft (stricter, top-5 codes only) or a
merged M2_complete+NLP version are better candidates.

In [ ]:
M2_complete_full = M2_complete.merge(nlp_for_merge, on='hadm_id', how='left')
nlp_cols_list = nlp_for_merge.columns.drop('hadm_id').tolist()
M2_complete_full[nlp_cols_list] = M2_complete_full[nlp_cols_list].fillna(0).astype(int)

print("M2_complete_full shape:", M2_complete_full.shape)
print(M2_complete_full.head())

M2_complete_full.to_csv('M2_complete_full.csv', index=False)
print("Saved M2_complete_full.csv")

Result:
-----
 M2_complete_full, 24,330 visits (3,749 patients, unchanged),
92 columns total: 2 IDs + 5 baseline features + 74 Z-codes + 11 NLP
columns.

Completing M3 with NLP Features
----

In [ ]:
M3_full = M3.merge(nlp_for_merge, on='hadm_id', how='left')
nlp_cols_list = nlp_for_merge.columns.drop('hadm_id').tolist()
M3_full[nlp_cols_list] = M3_full[nlp_cols_list].fillna(0).astype(int)

print("M3_full shape:", M3_full.shape)
print(M3_full.head())

M3_full.to_csv('M3_full.csv', index=False)
print("Saved M3_full.csv")

Result: M3_full, 20,061 visits (3,008 patients, unchanged), 23 columns
total: 2 IDs + 5 baseline features + 5 Z-codes (Z590, Z62810, Z560,
Z5900, Z602) + 11 NLP columns.

M3_full is now the most complete, most reliable SDOH matrix built so
far: smaller and denser than M2_complete_full (3,008 vs 3,749
patients), restricted to only the top 5 strongest SDOH signals, and
includes both structured diagnosis codes and discharge-note keyword
signal.


Random Forest on Every Matrix
------

Starting with the baseline

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score, roc_auc_score, roc_curve
import matplotlib.pyplot as plt

features_M0 = ['insurance', 'race', 'language', 'marital_status', 'anchor_age']
target = 'readmit_30_any'

X = M0[features_M0].copy()
y = M0[target]

# encode categorical columns as numbers
for col in ['insurance', 'race', 'language', 'marital_status']:
    X[col] = LabelEncoder().fit_transform(X[col].astype(str))

print(X.head())
print("\nTarget distribution:\n", y.value_counts())

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

rf_M0 = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')
rf_M0.fit(X_train, y_train)

y_pred_train = rf_M0.predict(X_train)
y_pred_test = rf_M0.predict(X_test)
y_proba_test = rf_M0.predict_proba(X_test)[:, 1]

print("=== M0 (Baseline Only) Random Forest Results ===")
print(f"Train Accuracy: {accuracy_score(y_train, y_pred_train):.4f}")
print(f"Test Accuracy:  {accuracy_score(y_test, y_pred_test):.4f}")
print(f"Recall:         {recall_score(y_test, y_pred_test):.4f}")
print(f"Precision:      {precision_score(y_test, y_pred_test):.4f}")
print(f"F1 Score:       {f1_score(y_test, y_pred_test):.4f}")
print(f"AUC:            {roc_auc_score(y_test, y_proba_test):.4f}")

fpr, tpr, thresholds = roc_curve(y_test, y_proba_test)
plt.figure(figsize=(7, 6))
plt.plot(fpr, tpr, label=f"AUC = {roc_auc_score(y_test, y_proba_test):.3f}")
plt.plot([0, 1], [0, 1], linestyle='--', color='gray', label='Random guess')
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve: M0 (Baseline Only)")
plt.legend()
plt.show()

In [ ]:
print(mimic_admissions.columns.tolist())

In [ ]:
missing_admissions = (mimic_admissions.isnull().sum() / len(mimic_admissions) * 100).round(2)
print(missing_admissions.sort_values(ascending=False))

In [ ]:
missing_patients = (mimic_patients.isnull().sum() / len(mimic_patients) * 100).round(2)
print("=== mimic_patients missing % ===")
print(missing_patients.sort_values(ascending=False))

missing_diagnoses = (mimic_diagnoses.isnull().sum() / len(mimic_diagnoses) * 100).round(2)
print("\n=== mimic_diagnoses missing % ===")
print(missing_diagnoses.sort_values(ascending=False))

In [ ]:
print("Unique patients in admissions table:", mimic_admissions['subject_id'].nunique())
print("Unique patients in patients table:", mimic_patients['subject_id'].nunique())
print("Unique patients in diagnoses table:", mimic_diagnoses['subject_id'].nunique())


In [ ]:
# confirm: are admissions patients a subset of the patients table?
admissions_patients = set(mimic_admissions['subject_id'].unique())
full_patients = set(mimic_patients['subject_id'].unique())

print("Patients in admissions but NOT in patients table:", len(admissions_patients - full_patients))
print("Patients in patients table but NOT in admissions:", len(full_patients - admissions_patients))

In [ ]:
print("Missing % for anchor_age in mimic_patients:", mimic_patients['anchor_age'].isnull().sum() / len(mimic_patients) * 100)
print("Missing % for anchor_age in final M0:", M0['anchor_age'].isnull().sum() / len(M0) * 100)

In [ ]:
demo_cols_full = ['insurance', 'race', 'language', 'marital_status']

missing_demo = mimic_admissions[demo_cols_full].isnull().sum() / len(mimic_admissions) * 100
missing_age = pd.Series({'anchor_age': mimic_patients['anchor_age'].isnull().sum() / len(mimic_patients) * 100})

missing_all_D0 = pd.concat([missing_demo, missing_age]).round(2)
print("Missing % for all 5 D0 features:")
print(missing_all_D0.sort_values(ascending=False))

In [ ]:
thresholds_to_test = [20, 15, 10, 8, 5, 3, 1]

threshold_table = pd.DataFrame({
    'threshold_%': thresholds_to_test,
    'groups_kept': [(group_coverage_clean_pct >= t).sum() for t in thresholds_to_test]
})

print(threshold_table)

In [ ]:
bins = [0, 5, 10, 15, 20, 100]
labels = ['0-5%', '5-10%', '10-15%', '15-20%', '20%+']

bucket_counts = pd.cut(group_coverage_clean_pct, bins=bins, labels=labels, right=False).value_counts().sort_index()
print("\nNumber of diagnosis groups by coverage bucket:")
print(bucket_counts)

In [ ]:
top33_groups = group_coverage_clean_pct.sort_values(ascending=False).head(33).index.tolist()

diag_dummies_33 = diag_grouped_clean[diag_grouped_clean['icd_group'].isin(top33_groups)]
diag_dummies_33 = pd.crosstab(diag_dummies_33['hadm_id'], diag_dummies_33['icd_group'])
diag_dummies_33 = (diag_dummies_33 > 0).astype(int).reset_index()

visits_with_33 = (diag_dummies_33[top33_groups].sum(axis=1) > 0).sum()
pct_33 = visits_with_33 / 546028 * 100

print(f"Visits with >=1 of top 33 diagnoses: {visits_with_33:,} ({pct_33:.1f}%)")
print(f"Visits with >=1 of top 15 diagnoses: 400,075 (73.3%)")
print(f"\nFeatures 'lost' by stopping at 15 instead of 33: {33-15}")
print(f"Coverage gained by going to 33: {pct_33 - 73.3:.1f} percentage points")

In [ ]:
sdoh_presence_table = pd.DataFrame({
    'patients_with_code': zcode_counts_cohort,
    'pct_of_cohort_3749': zcode_pct_cohort,
    'pct_missing_of_cohort': 100 - zcode_pct_cohort
}).sort_values('pct_of_cohort_3749', ascending=False)

print(sdoh_presence_table)
sdoh_presence_table.to_csv('sdoh_full_presence_table.csv')
print("\nSaved sdoh_full_presence_table.csv")

In [ ]:
pd.set_option('display.max_rows', 100)
print(sdoh_presence_table.to_string())

In [ ]:
sdoh_thresholds_to_test = [50, 20, 10, 8, 5, 3, 1]

sdoh_threshold_table = pd.DataFrame({
    'threshold_%': sdoh_thresholds_to_test,
    'zcodes_kept': [(zcode_pct_cohort >= t).sum() for t in sdoh_thresholds_to_test]
})
print(sdoh_threshold_table)

In [ ]:
bins = [0, 5, 10, 20, 50, 100]
labels = ['0-5%', '5-10%', '10-20%', '20-50%', '50%+']

sdoh_bucket_counts = pd.cut(zcode_pct_cohort, bins=bins, labels=labels, right=False).value_counts().sort_index()
print("\nNumber of SDOH Z-codes by coverage bucket:")
print(sdoh_bucket_counts)

In [ ]:
readmit_check = mimic_admissions[['subject_id', 'hadm_id', 'admittime', 'dischtime', 'admission_type']].copy()
readmit_check['admittime'] = pd.to_datetime(readmit_check['admittime'])
readmit_check['dischtime'] = pd.to_datetime(readmit_check['dischtime'])
readmit_check = readmit_check.sort_values(['subject_id', 'admittime']).reset_index(drop=True)

In [ ]:
readmit_check['next_admittime'] = readmit_check.groupby('subject_id')['admittime'].shift(-1)
readmit_check['next_admission_type'] = readmit_check.groupby('subject_id')['admission_type'].shift(-1)
readmit_check['days_to_next_admit'] = (readmit_check['next_admittime'] - readmit_check['dischtime']).dt.days

In [ ]:
readmit_check['readmit_30_any'] = (readmit_check['days_to_next_admit'] <= 30).astype(int)
readmit_check['readmit_30_same_type'] = (
    (readmit_check['days_to_next_admit'] <= 30) &
    (readmit_check['admission_type'] == readmit_check['next_admission_type'])
).astype(int)

In [ ]:
readmit_cols = ['hadm_id', 'readmit_30_any', 'readmit_30_same_type']
M0 = M0.merge(readmit_check[readmit_cols], on='hadm_id', how='left')
# ... same pattern for M1, M2_full_5pct, M2_full_10pct

In [ ]:
readmit_check = mimic_admissions[['subject_id', 'hadm_id', 'admittime', 'dischtime', 'admission_type']].copy()
readmit_check['admittime'] = pd.to_datetime(readmit_check['admittime'])
readmit_check['dischtime'] = pd.to_datetime(readmit_check['dischtime'])
readmit_check = readmit_check.sort_values(['subject_id', 'admittime']).reset_index(drop=True)
print(readmit_check.shape)
print(readmit_check.head())

In [ ]:
readmit_check['next_admittime'] = readmit_check.groupby('subject_id')['admittime'].shift(-1)
readmit_check['next_admission_type'] = readmit_check.groupby('subject_id')['admission_type'].shift(-1)
readmit_check['days_to_next_admit'] = (readmit_check['next_admittime'] - readmit_check['dischtime']).dt.days
print(readmit_check[['subject_id', 'hadm_id', 'dischtime', 'next_admittime', 'days_to_next_admit']].head(10))

In [ ]:
readmit_check['readmit_30_any'] = (readmit_check['days_to_next_admit'] <= 30).astype(int)
readmit_check['readmit_30_same_type'] = (
    (readmit_check['days_to_next_admit'] <= 30) &
    (readmit_check['admission_type'] == readmit_check['next_admission_type'])
).astype(int)
print(readmit_check['readmit_30_any'].value_counts())
print(readmit_check['readmit_30_same_type'].value_counts())

In [ ]:
# check if M0 already has the readmission columns
print(M0.columns.tolist())

In [ ]:
# rebuild a clean M0 from scratch
M0_clean_final = mimic_admissions[['subject_id', 'hadm_id', 'insurance', 'race', 'language', 'marital_status']].copy()
M0_clean_final[['insurance', 'race', 'language', 'marital_status']] = M0_clean_final[['insurance', 'race', 'language', 'marital_status']].fillna('Unknown')
M0_clean_final = M0_clean_final.merge(mimic_patients[['subject_id', 'anchor_age']], on='subject_id', how='left')

# merge readmission labels in once
M0_clean_final = M0_clean_final.merge(readmit_check[readmit_cols], on='hadm_id', how='left')

print("M0 shape:", M0_clean_final.shape)
print(M0_clean_final.columns.tolist())
print(M0_clean_final[['subject_id', 'hadm_id', 'readmit_30_any', 'readmit_30_same_type']].head())

In [ ]:
import matplotlib.pyplot as plt

labels = ['readmit_30_any', 'readmit_30_same_type']
readmitted = [111035, 44828]
not_readmitted = [434993, 501200]

x = range(len(labels))
plt.figure(figsize=(8, 6))
plt.bar(x, not_readmitted, label='Not Readmitted (0)', color='lightgray')
plt.bar(x, readmitted, bottom=not_readmitted, label='Readmitted (1)', color='#c0567a')
plt.xticks(x, labels)
plt.ylabel('Number of Visits')
plt.title('30-Day Readmission: Class Balance')
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
readmit_check['days_to_next_admit'].dropna().clip(upper=90).hist(bins=50)
plt.axvline(x=30, color='red', linestyle='--', label='30-day cutoff')
plt.xlabel('Days Until Next Admission')
plt.ylabel('Number of Visits')
plt.title('Distribution of Time to Next Admission (capped at 90 days)')
plt.legend()
plt.show()

In [ ]:
same_type = ((readmit_check['readmit_30_any'] == 1) & (readmit_check['readmit_30_same_type'] == 1)).sum()
diff_type = ((readmit_check['readmit_30_any'] == 1) & (readmit_check['readmit_30_same_type'] == 0)).sum()
not_readmitted = (readmit_check['readmit_30_any'] == 0).sum()

print(f"Readmitted, SAME type: {same_type:,}")
print(f"Readmitted, DIFFERENT type: {diff_type:,}")
print(f"NOT readmitted: {not_readmitted:,}")
print(f"Total: {same_type + diff_type + not_readmitted:,}")

In [ ]:
print(M1_15.columns.tolist())

In [ ]:
M1_15 = M1_15.merge(readmit_check[readmit_cols], on='hadm_id', how='left')
print("M1_15 shape:", M1_15.shape)
print(M1_15[['subject_id', 'hadm_id', 'readmit_30_any', 'readmit_30_same_type']].head())

In [ ]:
print("=== M1 readmission breakdown ===")
print(M1['readmit_30_any'].value_counts())
print(M1['readmit_30_any'].value_counts(normalize=True) * 100)

print("\n=== M1_15 readmission breakdown ===")
print(M1_15['readmit_30_any'].value_counts())
print(M1_15['readmit_30_any'].value_counts(normalize=True) * 100)

In [ ]:
print(M2_complete_full.columns.tolist())

In [ ]:
M2_complete_full = M2_complete_full.merge(readmit_check[readmit_cols], on='hadm_id', how='left')
print("M2_complete_full shape:", M2_complete_full.shape)
print(M2_complete_full[['subject_id', 'hadm_id', 'readmit_30_any', 'readmit_30_same_type']].head())

In [ ]:
print("=== M2_complete_full readmission breakdown ===")
same_type = ((M2_complete_full['readmit_30_any'] == 1) & (M2_complete_full['readmit_30_same_type'] == 1)).sum()
diff_type = ((M2_complete_full['readmit_30_any'] == 1) & (M2_complete_full['readmit_30_same_type'] == 0)).sum()
not_readmitted = (M2_complete_full['readmit_30_any'] == 0).sum()

print(f"Readmitted, SAME type: {same_type:,}")
print(f"Readmitted, DIFFERENT type: {diff_type:,}")
print(f"NOT readmitted: {not_readmitted:,}")
print(f"Total: {same_type + diff_type + not_readmitted:,}")

In [ ]:
import matplotlib.pyplot as plt

labels = ['Full Population\n(546,028 visits)', 'SDOH-Coded Cohort\n(24,330 visits)']
readmitted = [111035, 9395]
not_readmitted = [434993, 14935]

x = range(len(labels))
plt.figure(figsize=(8, 6))
plt.bar(x, not_readmitted, label='Not Readmitted (0)', color='lightgray')
plt.bar(x, readmitted, bottom=not_readmitted, label='Readmitted (1)', color='#c0567a')
plt.xticks(x, labels)
plt.ylabel('Number of Visits')
plt.title('30-Day Readmission Rate: Full Population vs. SDOH Cohort')
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(7, 6))
rates = [20.3, 38.6]
labels = ['Full Population\n(546,028 visits)', 'SDOH-Coded Cohort\n(24,330 visits)']
bars = plt.bar(labels, rates, color=['#8ba888', '#c0567a'])
plt.ylabel('% Readmitted Within 30 Days')
plt.title('Readmission Rate: Full Population vs. SDOH Cohort')
plt.ylim(0, 50)
for bar, rate in zip(bars, rates):
    plt.text(bar.get_x() + bar.get_width()/2, rate + 1, f'{rate}%', ha='center', fontweight='bold')
plt.show()

In [ ]:
plt.figure(figsize=(7, 6))
labels = ['Full Population', 'SDOH-Coded Cohort']
readmit_pct = [20.3, 38.6]
not_readmit_pct = [79.7, 61.4]

plt.bar(labels, not_readmit_pct, label='Not Readmitted (0)', color='lightgray')
plt.bar(labels, readmit_pct, bottom=not_readmit_pct, label='Readmitted (1)', color='#c0567a')
plt.ylabel('% of Visits')
plt.title('30-Day Readmission Rate (Normalized to 100%)')
plt.legend()
plt.show()

In [ ]:
print(M2_complete.columns.tolist())

In [ ]:
M2_complete = M2_complete.merge(readmit_check[readmit_cols], on='hadm_id', how='left')
print("M2_complete shape:", M2_complete.shape)
print(M2_complete[['subject_id', 'hadm_id', 'readmit_30_any', 'readmit_30_same_type']].head())

In [ ]:
try:
    print(M3_full.shape)
    print(M3_full.columns.tolist())
except NameError:
    print("M3_full does not exist — need to rebuild from scratch")

In [ ]:
M3_full = M3_full.merge(readmit_check[readmit_cols], on='hadm_id', how='left')
print("M3_full shape:", M3_full.shape)
print(M3_full[['subject_id', 'hadm_id', 'readmit_30_any', 'readmit_30_same_type']].head())

In [ ]:
try:
    print("M0 exists, shape:", M0.shape)
    print("Has readmit_30_any:", 'readmit_30_any' in M0.columns)
except NameError:
    print("M0 does not exist in memory — fresh session")

In [ ]:
print(M0.shape)
print(M0.columns.tolist())
M0.head()

In [ ]:
M0 = M0[['subject_id', 'hadm_id', 'insurance', 'race', 'language', 'marital_status', 'anchor_age',
          'readmit_30_any_x', 'readmit_30_same_type_x']].copy()
M0 = M0.rename(columns={'readmit_30_any_x': 'readmit_30_any', 'readmit_30_same_type_x': 'readmit_30_same_type'})

print(M0.shape)
print(M0.columns.tolist())

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score, roc_auc_score, roc_curve
import matplotlib.pyplot as plt

features_M0 = ['insurance', 'race', 'language', 'marital_status', 'anchor_age']
target = 'readmit_30_any'

X = M0[features_M0].copy()
y = M0[target]

# encode categorical columns as numbers
for col in ['insurance', 'race', 'language', 'marital_status']:
    X[col] = LabelEncoder().fit_transform(X[col].astype(str))

print(X.head())
print("\nTarget distribution:\n", y.value_counts())

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

rf_M0 = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')
rf_M0.fit(X_train, y_train)

y_pred_train = rf_M0.predict(X_train)
y_pred_test = rf_M0.predict(X_test)
y_proba_test = rf_M0.predict_proba(X_test)[:, 1]

print("=== M0 (Baseline Only) Random Forest Results ===")
print(f"Train Accuracy: {accuracy_score(y_train, y_pred_train):.4f}")
print(f"Test Accuracy:  {accuracy_score(y_test, y_pred_test):.4f}")
print(f"Recall:         {recall_score(y_test, y_pred_test):.4f}")
print(f"Precision:      {precision_score(y_test, y_pred_test):.4f}")
print(f"F1 Score:       {f1_score(y_test, y_pred_test):.4f}")
print(f"AUC:            {roc_auc_score(y_test, y_proba_test):.4f}")

fpr, tpr, thresholds = roc_curve(y_test, y_proba_test)
plt.figure(figsize=(7, 6))
plt.plot(fpr, tpr, label=f"AUC = {roc_auc_score(y_test, y_proba_test):.3f}")
plt.plot([0, 1], [0, 1], linestyle='--', color='gray', label='Random guess')
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve: M0 (Baseline Only)")
plt.legend()
plt.show()

In [ ]:
features_M1 = ['insurance', 'race', 'language', 'marital_status', 'anchor_age',
               '250', '272', '401', 'E11', 'E78', 'E87', 'I10', 'K21']
target = 'readmit_30_any'

X = M1[features_M1].copy()
y = M1[target]

for col in ['insurance', 'race', 'language', 'marital_status']:
    X[col] = LabelEncoder().fit_transform(X[col].astype(str))

print(X.head())
print("\nTarget distribution:\n", y.value_counts())

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

rf_M1 = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')
rf_M1.fit(X_train, y_train)

y_pred_train = rf_M1.predict(X_train)
y_pred_test = rf_M1.predict(X_test)
y_proba_test = rf_M1.predict_proba(X_test)[:, 1]

print("=== M1 (Baseline + Diagnosis) Random Forest Results ===")
print(f"Train Accuracy: {accuracy_score(y_train, y_pred_train):.4f}")
print(f"Test Accuracy:  {accuracy_score(y_test, y_pred_test):.4f}")
print(f"Recall:         {recall_score(y_test, y_pred_test):.4f}")
print(f"Precision:      {precision_score(y_test, y_pred_test):.4f}")
print(f"F1 Score:       {f1_score(y_test, y_pred_test):.4f}")
print(f"AUC:            {roc_auc_score(y_test, y_proba_test):.4f}")

fpr, tpr, thresholds = roc_curve(y_test, y_proba_test)
plt.figure(figsize=(7, 6))
plt.plot(fpr, tpr, label=f"AUC = {roc_auc_score(y_test, y_proba_test):.3f}")
plt.plot([0, 1], [0, 1], linestyle='--', color='gray', label='Random guess')
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve: M1 (Baseline + Diagnosis)")
plt.legend()
plt.show()

In [ ]:
print(M2_full_5pct.columns.tolist())

In [ ]:
features_M2_5pct = ['insurance', 'race', 'language', 'marital_status', 'anchor_age',
                     'Z590', 'Z62810', 'Z560', 'Z5900', 'Z602',
                     'homeless', 'alcohol_use', 'drug_use', 'substance_abuse', 'tobacco_use',
                     'lives_alone', 'unemployed', 'food_insecure', 'domestic_violence',
                     'limited_support', 'financial_stress']
target = 'readmit_30_any'

X = M2_full_5pct[features_M2_5pct].copy()
y = M2_full_5pct[target]

for col in ['insurance', 'race', 'language', 'marital_status']:
    X[col] = LabelEncoder().fit_transform(X[col].astype(str))

print(X.head())
print("\nTarget distribution:\n", y.value_counts())

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

rf_M2_5pct = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')
rf_M2_5pct.fit(X_train, y_train)

y_pred_train = rf_M2_5pct.predict(X_train)
y_pred_test = rf_M2_5pct.predict(X_test)
y_proba_test = rf_M2_5pct.predict_proba(X_test)[:, 1]

print("=== M2_full_5pct (Baseline + SDOH) Random Forest Results ===")
print(f"Train Accuracy: {accuracy_score(y_train, y_pred_train):.4f}")
print(f"Test Accuracy:  {accuracy_score(y_test, y_pred_test):.4f}")
print(f"Recall:         {recall_score(y_test, y_pred_test):.4f}")
print(f"Precision:      {precision_score(y_test, y_pred_test):.4f}")
print(f"F1 Score:       {f1_score(y_test, y_pred_test):.4f}")
print(f"AUC:            {roc_auc_score(y_test, y_proba_test):.4f}")

fpr, tpr, thresholds = roc_curve(y_test, y_proba_test)
plt.figure(figsize=(7, 6))
plt.plot(fpr, tpr, label=f"AUC = {roc_auc_score(y_test, y_proba_test):.3f}")
plt.plot([0, 1], [0, 1], linestyle='--', color='gray', label='Random guess')
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve: M2_full_5pct (Baseline + SDOH)")
plt.legend()
plt.show()

In [ ]:
print(M2_full_10pct.columns.tolist())

In [ ]:
features_M2_10pct = ['insurance', 'race', 'language', 'marital_status', 'anchor_age',
                      'Z590',
                      'homeless', 'alcohol_use', 'drug_use', 'substance_abuse', 'tobacco_use',
                      'lives_alone', 'unemployed', 'food_insecure', 'domestic_violence',
                      'limited_support', 'financial_stress']
target = 'readmit_30_any'

X = M2_full_10pct[features_M2_10pct].copy()
y = M2_full_10pct[target]

for col in ['insurance', 'race', 'language', 'marital_status']:
    X[col] = LabelEncoder().fit_transform(X[col].astype(str))

print(X.head())
print("\nTarget distribution:\n", y.value_counts())

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

rf_M2_10pct = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')
rf_M2_10pct.fit(X_train, y_train)

y_pred_train = rf_M2_10pct.predict(X_train)
y_pred_test = rf_M2_10pct.predict(X_test)
y_proba_test = rf_M2_10pct.predict_proba(X_test)[:, 1]

print("=== M2_full_10pct (Baseline + SDOH, strict) Random Forest Results ===")
print(f"Train Accuracy: {accuracy_score(y_train, y_pred_train):.4f}")
print(f"Test Accuracy:  {accuracy_score(y_test, y_pred_test):.4f}")
print(f"Recall:         {recall_score(y_test, y_pred_test):.4f}")
print(f"Precision:      {precision_score(y_test, y_pred_test):.4f}")
print(f"F1 Score:       {f1_score(y_test, y_pred_test):.4f}")
print(f"AUC:            {roc_auc_score(y_test, y_proba_test):.4f}")

fpr, tpr, thresholds = roc_curve(y_test, y_proba_test)
plt.figure(figsize=(7, 6))
plt.plot(fpr, tpr, label=f"AUC = {roc_auc_score(y_test, y_proba_test):.3f}")
plt.plot([0, 1], [0, 1], linestyle='--', color='gray', label='Random guess')
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve: M2_full_10pct (Baseline + SDOH, strict)")
plt.legend()
plt.show()

In [ ]:
print(M2_complete_full.columns.tolist())


In [ ]:
exclude_cols = ['subject_id', 'hadm_id', 'readmit_30_any', 'readmit_30_same_type']
features_M2_complete = [c for c in M2_complete_full.columns if c not in exclude_cols]
target = 'readmit_30_any'

X = M2_complete_full[features_M2_complete].copy()
y = M2_complete_full[target]

for col in ['insurance', 'race', 'language', 'marital_status']:
    X[col] = LabelEncoder().fit_transform(X[col].astype(str))

print(X.shape)
print("\nTarget distribution:\n", y.value_counts())

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

rf_M2_complete = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')
rf_M2_complete.fit(X_train, y_train)

y_pred_train = rf_M2_complete.predict(X_train)
y_pred_test = rf_M2_complete.predict(X_test)
y_proba_test = rf_M2_complete.predict_proba(X_test)[:, 1]

print("=== M2_complete_full (SDOH Cohort Only) Random Forest Results ===")
print(f"Train Accuracy: {accuracy_score(y_train, y_pred_train):.4f}")
print(f"Test Accuracy:  {accuracy_score(y_test, y_pred_test):.4f}")
print(f"Recall:         {recall_score(y_test, y_pred_test):.4f}")
print(f"Precision:      {precision_score(y_test, y_pred_test):.4f}")
print(f"F1 Score:       {f1_score(y_test, y_pred_test):.4f}")
print(f"AUC:            {roc_auc_score(y_test, y_proba_test):.4f}")

fpr, tpr, thresholds = roc_curve(y_test, y_proba_test)
plt.figure(figsize=(7, 6))
plt.plot(fpr, tpr, label=f"AUC = {roc_auc_score(y_test, y_proba_test):.3f}")
plt.plot([0, 1], [0, 1], linestyle='--', color='gray', label='Random guess')
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve: M2_complete_full (SDOH Cohort Only)")
plt.legend()
plt.show()

In [ ]:
print(M3_full.columns.tolist())

In [ ]:
features_M3 = ['insurance', 'race', 'language', 'marital_status', 'anchor_age',
               'Z590', 'Z62810', 'Z560', 'Z5900', 'Z602',
               'homeless', 'alcohol_use', 'drug_use', 'substance_abuse', 'tobacco_use',
               'lives_alone', 'unemployed', 'food_insecure', 'domestic_violence',
               'limited_support', 'financial_stress']
target = 'readmit_30_any'

X = M3_full[features_M3].copy()
y = M3_full[target]

for col in ['insurance', 'race', 'language', 'marital_status']:
    X[col] = LabelEncoder().fit_transform(X[col].astype(str))

print(X.shape)
print("\nTarget distribution:\n", y.value_counts())

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

rf_M3 = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')
rf_M3.fit(X_train, y_train)

y_pred_train = rf_M3.predict(X_train)
y_pred_test = rf_M3.predict(X_test)
y_proba_test = rf_M3.predict_proba(X_test)[:, 1]

print("=== M3_full (Top-5 SDOH Cohort) Random Forest Results ===")
print(f"Train Accuracy: {accuracy_score(y_train, y_pred_train):.4f}")
print(f"Test Accuracy:  {accuracy_score(y_test, y_pred_test):.4f}")
print(f"Recall:         {recall_score(y_test, y_pred_test):.4f}")
print(f"Precision:      {precision_score(y_test, y_pred_test):.4f}")
print(f"F1 Score:       {f1_score(y_test, y_pred_test):.4f}")
print(f"AUC:            {roc_auc_score(y_test, y_proba_test):.4f}")

fpr, tpr, thresholds = roc_curve(y_test, y_proba_test)
plt.figure(figsize=(7, 6))
plt.plot(fpr, tpr, label=f"AUC = {roc_auc_score(y_test, y_proba_test):.3f}")
plt.plot([0, 1], [0, 1], linestyle='--', color='gray', label='Random guess')
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve: M3_full (Top-5 SDOH Cohort)")
plt.legend()
plt.show()

Feature-Group Experiments
-----------

In [ ]:
diag_cols = ['250', '272', '401', 'E11', 'E78', 'E87', 'I10', 'K21']
M_EHR_only = M1[['subject_id', 'hadm_id'] + diag_cols + ['readmit_30_any', 'readmit_30_same_type']].copy()

print("EHR only shape:", M_EHR_only.shape)
print(M_EHR_only.columns.tolist())
M_EHR_only.to_csv('M_EHR_only.csv', index=False)
print("Saved M_EHR_only.csv")

In [ ]:
features_EHR_only = ['250', '272', '401', 'E11', 'E78', 'E87', 'I10', 'K21']
target = 'readmit_30_any'

X = M_EHR_only[features_EHR_only].copy()
y = M_EHR_only[target]

print(X.head())
print("\nTarget distribution:\n", y.value_counts())

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

rf_EHR_only = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')
rf_EHR_only.fit(X_train, y_train)

y_pred_train = rf_EHR_only.predict(X_train)
y_pred_test = rf_EHR_only.predict(X_test)
y_proba_test = rf_EHR_only.predict_proba(X_test)[:, 1]

print("=== EHR Only Random Forest Results ===")
print(f"Train Accuracy: {accuracy_score(y_train, y_pred_train):.4f}")
print(f"Test Accuracy:  {accuracy_score(y_test, y_pred_test):.4f}")
print(f"Recall:         {recall_score(y_test, y_pred_test):.4f}")
print(f"Precision:      {precision_score(y_test, y_pred_test):.4f}")
print(f"F1 Score:       {f1_score(y_test, y_pred_test):.4f}")
print(f"AUC:            {roc_auc_score(y_test, y_proba_test):.4f}")

fpr, tpr, thresholds = roc_curve(y_test, y_proba_test)
plt.figure(figsize=(7, 6))
plt.plot(fpr, tpr, label=f"AUC = {roc_auc_score(y_test, y_proba_test):.3f}")
plt.plot([0, 1], [0, 1], linestyle='--', color='gray', label='Random guess')
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve: EHR Only")
plt.legend()
plt.show()

In [ ]:
sdoh_cols = ['Z590', 'Z62810', 'Z560', 'Z5900', 'Z602',
             'homeless', 'alcohol_use', 'drug_use', 'substance_abuse', 'tobacco_use',
             'lives_alone', 'unemployed', 'food_insecure', 'domestic_violence',
             'limited_support', 'financial_stress']

M_SDOH_only = M2_full_5pct[['subject_id', 'hadm_id'] + sdoh_cols + ['readmit_30_any', 'readmit_30_same_type']].copy()

print("SDOH only shape:", M_SDOH_only.shape)
print(M_SDOH_only.columns.tolist())
M_SDOH_only.to_csv('M_SDOH_only.csv', index=False)
print("Saved M_SDOH_only.csv")

In [ ]:
features_SDOH_only = ['Z590', 'Z62810', 'Z560', 'Z5900', 'Z602',
                       'homeless', 'alcohol_use', 'drug_use', 'substance_abuse', 'tobacco_use',
                       'lives_alone', 'unemployed', 'food_insecure', 'domestic_violence',
                       'limited_support', 'financial_stress']
target = 'readmit_30_any'

X = M_SDOH_only[features_SDOH_only].copy()
y = M_SDOH_only[target]

# no encoding needed — all columns already 0/1

print(X.head())
print("\nTarget distribution:\n", y.value_counts())

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

rf_SDOH_only = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')
rf_SDOH_only.fit(X_train, y_train)

y_pred_train = rf_SDOH_only.predict(X_train)
y_pred_test = rf_SDOH_only.predict(X_test)
y_proba_test = rf_SDOH_only.predict_proba(X_test)[:, 1]

print("=== SDOH Only Random Forest Results ===")
print(f"Train Accuracy: {accuracy_score(y_train, y_pred_train):.4f}")
print(f"Test Accuracy:  {accuracy_score(y_test, y_pred_test):.4f}")
print(f"Recall:         {recall_score(y_test, y_pred_test):.4f}")
print(f"Precision:      {precision_score(y_test, y_pred_test):.4f}")
print(f"F1 Score:       {f1_score(y_test, y_pred_test):.4f}")
print(f"AUC:            {roc_auc_score(y_test, y_proba_test):.4f}")

fpr, tpr, thresholds = roc_curve(y_test, y_proba_test)
plt.figure(figsize=(7, 6))
plt.plot(fpr, tpr, label=f"AUC = {roc_auc_score(y_test, y_proba_test):.3f}")
plt.plot([0, 1], [0, 1], linestyle='--', color='gray', label='Random guess')
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve: SDOH Only")
plt.legend()
plt.show()

In [ ]:
diag_cols = ['250', '272', '401', 'E11', 'E78', 'E87', 'I10', 'K21']
sdoh_cols = ['Z590', 'Z62810', 'Z560', 'Z5900', 'Z602',
             'homeless', 'alcohol_use', 'drug_use', 'substance_abuse', 'tobacco_use',
             'lives_alone', 'unemployed', 'food_insecure', 'domestic_violence',
             'limited_support', 'financial_stress']

M_EHR_SDOH = M1[['subject_id', 'hadm_id'] + diag_cols].merge(
    M2_full_5pct[['subject_id', 'hadm_id'] + sdoh_cols + ['readmit_30_any', 'readmit_30_same_type']],
    on=['subject_id', 'hadm_id'],
    how='inner'
)

print("EHR + SDOH shape:", M_EHR_SDOH.shape)
print(M_EHR_SDOH.columns.tolist())
M_EHR_SDOH.to_csv('M_EHR_SDOH.csv', index=False)
print("Saved M_EHR_SDOH.csv")

In [ ]:
diag_cols = ['250', '272', '401', 'E11', 'E78', 'E87', 'I10', 'K21']
sdoh_cols = ['Z590', 'Z62810', 'Z560', 'Z5900', 'Z602',
             'homeless', 'alcohol_use', 'drug_use', 'substance_abuse', 'tobacco_use',
             'lives_alone', 'unemployed', 'food_insecure', 'domestic_violence',
             'limited_support', 'financial_stress']
features_EHR_SDOH = diag_cols + sdoh_cols
target = 'readmit_30_any'

X = M_EHR_SDOH[features_EHR_SDOH].copy()
y = M_EHR_SDOH[target]

# no encoding needed — all columns already 0/1

print(X.shape)
print("\nTarget distribution:\n", y.value_counts())

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

rf_EHR_SDOH = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')
rf_EHR_SDOH.fit(X_train, y_train)

y_pred_train = rf_EHR_SDOH.predict(X_train)
y_pred_test = rf_EHR_SDOH.predict(X_test)
y_proba_test = rf_EHR_SDOH.predict_proba(X_test)[:, 1]

print("=== EHR + SDOH Random Forest Results ===")
print(f"Train Accuracy: {accuracy_score(y_train, y_pred_train):.4f}")
print(f"Test Accuracy:  {accuracy_score(y_test, y_pred_test):.4f}")
print(f"Recall:         {recall_score(y_test, y_pred_test):.4f}")
print(f"Precision:      {precision_score(y_test, y_pred_test):.4f}")
print(f"F1 Score:       {f1_score(y_test, y_pred_test):.4f}")
print(f"AUC:            {roc_auc_score(y_test, y_proba_test):.4f}")

fpr, tpr, thresholds = roc_curve(y_test, y_proba_test)
plt.figure(figsize=(7, 6))
plt.plot(fpr, tpr, label=f"AUC = {roc_auc_score(y_test, y_proba_test):.3f}")
plt.plot([0, 1], [0, 1], linestyle='--', color='gray', label='Random guess')
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve: EHR + SDOH")
plt.legend()
plt.show()

In [ ]:
diag_cols = ['250', '272', '401', 'E11', 'E78', 'E87', 'I10', 'K21']
sdoh_cols = ['Z590', 'Z62810', 'Z560', 'Z5900', 'Z602',
             'homeless', 'alcohol_use', 'drug_use', 'substance_abuse', 'tobacco_use',
             'lives_alone', 'unemployed', 'food_insecure', 'domestic_violence',
             'limited_support', 'financial_stress']
baseline_cols = ['insurance', 'race', 'language', 'marital_status', 'anchor_age']

M_full_combined = M0[['subject_id', 'hadm_id'] + baseline_cols].merge(
    M1[['subject_id', 'hadm_id'] + diag_cols],
    on=['subject_id', 'hadm_id'], how='inner'
).merge(
    M2_full_5pct[['subject_id', 'hadm_id'] + sdoh_cols + ['readmit_30_any', 'readmit_30_same_type']],
    on=['subject_id', 'hadm_id'], how='inner'
)

print("Full combined shape:", M_full_combined.shape)
print(M_full_combined.columns.tolist())
M_full_combined.to_csv('M_full_combined.csv', index=False)
print("Saved M_full_combined.csv")

In [ ]:
baseline_cols = ['insurance', 'race', 'language', 'marital_status', 'anchor_age']
diag_cols = ['250', '272', '401', 'E11', 'E78', 'E87', 'I10', 'K21']
sdoh_cols = ['Z590', 'Z62810', 'Z560', 'Z5900', 'Z602',
             'homeless', 'alcohol_use', 'drug_use', 'substance_abuse', 'tobacco_use',
             'lives_alone', 'unemployed', 'food_insecure', 'domestic_violence',
             'limited_support', 'financial_stress']
features_full = baseline_cols + diag_cols + sdoh_cols
target = 'readmit_30_any'

X = M_full_combined[features_full].copy()
y = M_full_combined[target]

for col in ['insurance', 'race', 'language', 'marital_status']:
    X[col] = LabelEncoder().fit_transform(X[col].astype(str))

print(X.shape)
print("\nTarget distribution:\n", y.value_counts())

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

rf_full = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')
rf_full.fit(X_train, y_train)

y_pred_train = rf_full.predict(X_train)
y_pred_test = rf_full.predict(X_test)
y_proba_test = rf_full.predict_proba(X_test)[:, 1]

print("=== Baseline + EHR + SDOH (Full Combined) Random Forest Results ===")
print(f"Train Accuracy: {accuracy_score(y_train, y_pred_train):.4f}")
print(f"Test Accuracy:  {accuracy_score(y_test, y_pred_test):.4f}")
print(f"Recall:         {recall_score(y_test, y_pred_test):.4f}")
print(f"Precision:      {precision_score(y_test, y_pred_test):.4f}")
print(f"F1 Score:       {f1_score(y_test, y_pred_test):.4f}")
print(f"AUC:            {roc_auc_score(y_test, y_proba_test):.4f}")

fpr, tpr, thresholds = roc_curve(y_test, y_proba_test)
plt.figure(figsize=(7, 6))
plt.plot(fpr, tpr, label=f"AUC = {roc_auc_score(y_test, y_proba_test):.3f}")
plt.plot([0, 1], [0, 1], linestyle='--', color='gray', label='Random guess')
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve: Baseline + EHR + SDOH")
plt.legend()
plt.show()

Confusion Matrices
-------

In [ ]:
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

# Column groups
baseline_cols = ['insurance', 'race', 'language', 'marital_status', 'anchor_age']
diag_cols = ['250', '272', '401', 'E11', 'E78', 'E87', 'I10', 'K21']
sdoh_cols_5pct = ['Z590', 'Z62810', 'Z560', 'Z5900', 'Z602',
                  'homeless', 'alcohol_use', 'drug_use', 'substance_abuse', 'tobacco_use',
                  'lives_alone', 'unemployed', 'food_insecure', 'domestic_violence',
                  'limited_support', 'financial_stress']

exclude_cols_full = ['subject_id', 'hadm_id', 'readmit_30_any', 'readmit_30_same_type']
features_M2_complete_full_list = [c for c in M2_complete_full.columns if c not in exclude_cols_full]

experiments = {
    "M0 (Baseline)": (M0, baseline_cols),
    "EHR only": (M_EHR_only, diag_cols),
    "SDOH only": (M_SDOH_only, sdoh_cols_5pct),
    "M1 (Baseline+EHR)": (M1, baseline_cols + diag_cols),
    "M2_complete_full (Best)": (M2_complete_full, features_M2_complete_full_list),
    "M3_full (Top-5 SDOH)": (M3_full, baseline_cols + sdoh_cols_5pct),
    "EHR + SDOH": (M_EHR_SDOH, diag_cols + sdoh_cols_5pct),
    "Full Combined": (M_full_combined, baseline_cols + diag_cols + sdoh_cols_5pct),
}

target = 'readmit_30_any'
fig, axes = plt.subplots(2, 4, figsize=(22, 10))
axes = axes.flatten()

results_summary = []

for i, (name, (df, features)) in enumerate(experiments.items()):
    X = df[features].copy()
    y = df[target]
    for col in ['insurance', 'race', 'language', 'marital_status']:
        if col in X.columns:
            X[col] = LabelEncoder().fit_transform(X[col].astype(str))

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
    rf = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')
    rf.fit(X_train, y_train)
    y_pred = rf.predict(X_test)

    cm = confusion_matrix(y_test, y_pred)
    tn, fp, fn, tp = cm.ravel()
    results_summary.append({
        'Model': name, 'Total': len(y_test), 'Actual_Readmits': int(y_test.sum()),
        'TP': tp, 'FN': fn, 'TN': tn, 'FP': fp
    })

    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[i],
                xticklabels=['Pred: No', 'Pred: Yes'], yticklabels=['Actual: No', 'Actual: Yes'])
    axes[i].set_title(name, fontsize=11)

plt.tight_layout()
plt.savefig('all_confusion_matrices.png', dpi=150)
plt.show()

summary_df = pd.DataFrame(results_summary)
print(summary_df.to_string(index=False))
summary_df.to_csv('confusion_matrix_summary.csv', index=False)
print("\nSaved all_confusion_matrices.png and confusion_matrix_summary.csv")

In [ ]:
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

baseline_cols = ['insurance', 'race', 'language', 'marital_status', 'anchor_age']
sdoh_5pct_cols = ['Z590', 'Z62810', 'Z560', 'Z5900', 'Z602',
                  'homeless', 'alcohol_use', 'drug_use', 'substance_abuse', 'tobacco_use',
                  'lives_alone', 'unemployed', 'food_insecure', 'domestic_violence',
                  'limited_support', 'financial_stress']
sdoh_10pct_cols = ['Z590',
                    'homeless', 'alcohol_use', 'drug_use', 'substance_abuse', 'tobacco_use',
                    'lives_alone', 'unemployed', 'food_insecure', 'domestic_violence',
                    'limited_support', 'financial_stress']

target = 'readmit_30_any'

experiments = {
    "M2_full_5pct (5 Z-codes)": (M2_full_5pct, baseline_cols + sdoh_5pct_cols),
    "M2_full_10pct (1 Z-code)": (M2_full_10pct, baseline_cols + sdoh_10pct_cols),
}

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
results_summary = []

for i, (name, (df, features)) in enumerate(experiments.items()):
    X = df[features].copy()
    y = df[target]
    for col in ['insurance', 'race', 'language', 'marital_status']:
        if col in X.columns:
            X[col] = LabelEncoder().fit_transform(X[col].astype(str))

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
    rf = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')
    rf.fit(X_train, y_train)
    y_pred = rf.predict(X_test)

    cm = confusion_matrix(y_test, y_pred)
    tn, fp, fn, tp = cm.ravel()
    results_summary.append({
        'Model': name, 'Total': len(y_test), 'Actual_Readmits': int(y_test.sum()),
        'TP': tp, 'FN': fn, 'TN': tn, 'FP': fp
    })

    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[i],
                xticklabels=['Pred: No', 'Pred: Yes'], yticklabels=['Actual: No', 'Actual: Yes'])
    axes[i].set_title(name, fontsize=11)

plt.tight_layout()
plt.savefig('m2_5pct_10pct_confusion_matrices.png', dpi=150)
plt.show()

summary_df = pd.DataFrame(results_summary)
print(summary_df.to_string(index=False))
summary_df.to_csv('m2_5pct_10pct_confusion_summary.csv', index=False)
print("\nSaved m2_5pct_10pct_confusion_matrices.png")

In [ ]:
importances = pd.Series(rf_M2_complete.feature_importances_, index=features_M2_complete).sort_values(ascending=False)

print("Top 15 most important features:")
print(importances.head(15))

plt.figure(figsize=(8, 6))
importances.head(15).plot(kind='barh')
plt.title('Top 15 Feature Importances: M2_complete_full')
plt.xlabel('Importance')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

cluster_cols = ['insurance', 'race', 'language', 'marital_status', 'anchor_age',
                 '250', '272', '401', 'E11', 'E78', 'E87', 'I10', 'K21']

X_cluster = M1[cluster_cols].copy()
for col in ['insurance', 'race', 'language', 'marital_status']:
    X_cluster[col] = LabelEncoder().fit_transform(X_cluster[col].astype(str))

# Standardize since K-means is distance-based and anchor_age has a much larger scale than binary flags
scaler = StandardScaler()
X_cluster_scaled = scaler.fit_transform(X_cluster)

print("Cluster feature matrix shape:", X_cluster_scaled.shape)

In [ ]:
kmeans = KMeans(n_clusters=5, random_state=42, n_init=10)
cluster_labels = kmeans.fit_predict(X_cluster_scaled)

M1_clustered = M1.copy()
M1_clustered['cluster'] = cluster_labels

print("Cluster sizes:")
print(M1_clustered['cluster'].value_counts().sort_index())

In [ ]:
sdoh_patient_ids = M2_complete_full['subject_id'].unique()

M1_clustered['has_sdoh'] = M1_clustered['subject_id'].isin(sdoh_patient_ids).astype(int)

print("Cluster distribution among SDOH-coded patients:")
sdoh_cluster_dist = M1_clustered[M1_clustered['has_sdoh'] == 1]['cluster'].value_counts().sort_index()
print(sdoh_cluster_dist)

print("\nCluster distribution among ALL patients (for comparison):")
all_cluster_dist = M1_clustered['cluster'].value_counts().sort_index()
print(all_cluster_dist)

print("\n% of each cluster that has documented SDOH:")
pct_sdoh_per_cluster = (sdoh_cluster_dist / all_cluster_dist * 100).round(2)
print(pct_sdoh_per_cluster)

plt.figure(figsize=(8, 5))
pct_sdoh_per_cluster.plot(kind='bar')
plt.title('% of Patients with Documented SDOH, by Cluster')
plt.xlabel('Cluster')
plt.ylabel('% with SDOH documented')
plt.show()

In [ ]:
inertias = []
k_range = range(2, 11)

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_cluster_scaled)
    inertias.append(km.inertia_)
    print(f"k={k}, inertia={km.inertia_:.0f}")

plt.figure(figsize=(8, 5))
plt.plot(list(k_range), inertias, marker='o')
plt.xlabel('Number of clusters (k)')
plt.ylabel('Inertia')
plt.title('Elbow Method: Choosing Optimal k')
plt.show()

In [ ]:
cluster_profile = M1_clustered.groupby('cluster').agg({
    'anchor_age': 'mean',
    'insurance': lambda x: x.mode()[0],
    'race': lambda x: x.mode()[0],
    'marital_status': lambda x: x.mode()[0],
    'language': lambda x: x.mode()[0],
    '250': 'mean', '272': 'mean', '401': 'mean', 'E11': 'mean',
    'E78': 'mean', 'E87': 'mean', 'I10': 'mean', 'K21': 'mean'
})
print(cluster_profile.T)

In [ ]:
readmit_check = mimic_admissions[['subject_id', 'hadm_id', 'admittime', 'dischtime', 'admission_type']].copy()
readmit_check['admittime'] = pd.to_datetime(readmit_check['admittime'])
readmit_check['dischtime'] = pd.to_datetime(readmit_check['dischtime'])
readmit_check = readmit_check.sort_values(['subject_id', 'admittime']).reset_index(drop=True)

readmit_check['next_admittime'] = readmit_check.groupby('subject_id')['admittime'].shift(-1)
readmit_check['days_to_next_admit'] = (readmit_check['next_admittime'] - readmit_check['dischtime']).dt.days

readmit_check['readmit_60_any'] = (readmit_check['days_to_next_admit'] <= 60).astype(int)
readmit_check['readmit_90_any'] = (readmit_check['days_to_next_admit'] <= 90).astype(int)

print("30-day positive rate:", (readmit_check['days_to_next_admit'] <= 30).mean())
print("60-day positive rate:", readmit_check['readmit_60_any'].mean())
print("90-day positive rate:", readmit_check['readmit_90_any'].mean())

In [ ]:
M0_windows = M0[['subject_id', 'hadm_id', 'insurance', 'race', 'language', 'marital_status', 'anchor_age']].merge(
    readmit_check[['hadm_id', 'readmit_60_any', 'readmit_90_any']], on='hadm_id', how='left'
)
M0_windows[['readmit_60_any', 'readmit_90_any']] = M0_windows[['readmit_60_any', 'readmit_90_any']].fillna(0).astype(int)

print(M0_windows.shape)
print(M0_windows.columns.tolist())

In [ ]:
features_M0 = ['insurance', 'race', 'language', 'marital_status', 'anchor_age']

results_windows = []

for window in ['readmit_60_any', 'readmit_90_any']:
    X = M0_windows[features_M0].copy()
    y = M0_windows[window]
    for col in ['insurance', 'race', 'language', 'marital_status']:
        X[col] = LabelEncoder().fit_transform(X[col].astype(str))

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
    rf = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')
    rf.fit(X_train, y_train)

    y_pred_train = rf.predict(X_train)
    y_pred_test = rf.predict(X_test)
    y_proba_test = rf.predict_proba(X_test)[:, 1]

    print(f"=== M0 — {window} ===")
    print(f"Train Accuracy: {accuracy_score(y_train, y_pred_train):.4f}")
    print(f"Test Accuracy:  {accuracy_score(y_test, y_pred_test):.4f}")
    print(f"Recall:         {recall_score(y_test, y_pred_test):.4f}")
    print(f"Precision:      {precision_score(y_test, y_pred_test):.4f}")
    print(f"F1 Score:       {f1_score(y_test, y_pred_test):.4f}")
    print(f"AUC:            {roc_auc_score(y_test, y_proba_test):.4f}")
    print()

    results_windows.append({
        'Window': window, 'Train_Acc': accuracy_score(y_train, y_pred_train),
        'Test_Acc': accuracy_score(y_test, y_pred_test), 'Recall': recall_score(y_test, y_pred_test),
        'Precision': precision_score(y_test, y_pred_test), 'F1': f1_score(y_test, y_pred_test),
        'AUC': roc_auc_score(y_test, y_proba_test)
    })

windows_df = pd.DataFrame(results_windows)
print(windows_df.to_string(index=False))

In [ ]:
exclude_cols = ['subject_id', 'hadm_id', 'readmit_30_any', 'readmit_30_same_type']
features_M2_complete = [c for c in M2_complete_full.columns if c not in exclude_cols]

M2_complete_full_windows = M2_complete_full.merge(
    readmit_check[['hadm_id', 'readmit_60_any', 'readmit_90_any']], on='hadm_id', how='left'
)
M2_complete_full_windows[['readmit_60_any', 'readmit_90_any']] = M2_complete_full_windows[['readmit_60_any', 'readmit_90_any']].fillna(0).astype(int)

baseline_cols = ['insurance', 'race', 'language', 'marital_status', 'anchor_age']
diag_cols = ['250', '272', '401', 'E11', 'E78', 'E87', 'I10', 'K21']
sdoh_cols = ['Z590', 'Z62810', 'Z560', 'Z5900', 'Z602',
             'homeless', 'alcohol_use', 'drug_use', 'substance_abuse', 'tobacco_use',
             'lives_alone', 'unemployed', 'food_insecure', 'domestic_violence',
             'limited_support', 'financial_stress']

M_full_combined_windows = M_full_combined.merge(
    readmit_check[['hadm_id', 'readmit_60_any', 'readmit_90_any']], on='hadm_id', how='left'
)
M_full_combined_windows[['readmit_60_any', 'readmit_90_any']] = M_full_combined_windows[['readmit_60_any', 'readmit_90_any']].fillna(0).astype(int)

matrices_to_test = {
    'M2_complete_full': (M2_complete_full_windows, features_M2_complete),
    'Full Combined': (M_full_combined_windows, baseline_cols + diag_cols + sdoh_cols),
}

for name, (df, features) in matrices_to_test.items():
    for window in ['readmit_60_any', 'readmit_90_any']:
        X = df[features].copy()
        y = df[window]
        for col in ['insurance', 'race', 'language', 'marital_status']:
            if col in X.columns:
                X[col] = LabelEncoder().fit_transform(X[col].astype(str))

        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
        rf = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')
        rf.fit(X_train, y_train)
        y_proba_test = rf.predict_proba(X_test)[:, 1]
        y_pred_test = rf.predict(X_test)

        results_windows.append({
            'Matrix': name, 'Window': window,
            'Test_Acc': accuracy_score(y_test, y_pred_test), 'Recall': recall_score(y_test, y_pred_test),
            'Precision': precision_score(y_test, y_pred_test), 'F1': f1_score(y_test, y_pred_test),
            'AUC': roc_auc_score(y_test, y_proba_test)
        })

windows_df = pd.DataFrame(results_windows)
print(windows_df.to_string(index=False))
windows_df.to_csv('readmission_window_comparison.csv', index=False)

In [ ]:
exclude_cols = ['subject_id', 'hadm_id', 'readmit_30_any', 'readmit_30_same_type']
features_M2_complete = [c for c in M2_complete_full.columns if c not in exclude_cols]

M2_complete_full_windows = M2_complete_full.merge(
    readmit_check[['hadm_id', 'readmit_60_any', 'readmit_90_any']], on='hadm_id', how='left'
)
M2_complete_full_windows[['readmit_60_any', 'readmit_90_any']] = M2_complete_full_windows[['readmit_60_any', 'readmit_90_any']].fillna(0).astype(int)

baseline_cols = ['insurance', 'race', 'language', 'marital_status', 'anchor_age']
diag_cols = ['250', '272', '401', 'E11', 'E78', 'E87', 'I10', 'K21']
sdoh_cols = ['Z590', 'Z62810', 'Z560', 'Z5900', 'Z602',
             'homeless', 'alcohol_use', 'drug_use', 'substance_abuse', 'tobacco_use',
             'lives_alone', 'unemployed', 'food_insecure', 'domestic_violence',
             'limited_support', 'financial_stress']

M_full_combined_windows = M_full_combined.merge(
    readmit_check[['hadm_id', 'readmit_60_any', 'readmit_90_any']], on='hadm_id', how='left'
)
M_full_combined_windows[['readmit_60_any', 'readmit_90_any']] = M_full_combined_windows[['readmit_60_any', 'readmit_90_any']].fillna(0).astype(int)

matrices_to_test = {
    'M0': (M0_windows, features_M0),
    'M2_complete_full': (M2_complete_full_windows, features_M2_complete),
    'Full Combined': (M_full_combined_windows, baseline_cols + diag_cols + sdoh_cols),
}

results_windows = []
fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))

for i, (name, (df, features)) in enumerate(matrices_to_test.items()):
    ax = axes[i]
    for window, color in zip(['readmit_60_any', 'readmit_90_any'], ['tab:blue', 'tab:orange']):
        X = df[features].copy()
        y = df[window]
        for col in ['insurance', 'race', 'language', 'marital_status']:
            if col in X.columns:
                X[col] = LabelEncoder().fit_transform(X[col].astype(str))

        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
        rf = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')
        rf.fit(X_train, y_train)
        y_proba_test = rf.predict_proba(X_test)[:, 1]
        y_pred_test = rf.predict(X_test)

        auc = roc_auc_score(y_test, y_proba_test)
        fpr, tpr, _ = roc_curve(y_test, y_proba_test)
        ax.plot(fpr, tpr, label=f"{window.replace('readmit_','').replace('_any','')}-day (AUC={auc:.3f})", color=color)

        results_windows.append({
            'Matrix': name, 'Window': window,
            'Test_Acc': accuracy_score(y_test, y_pred_test), 'Recall': recall_score(y_test, y_pred_test),
            'Precision': precision_score(y_test, y_pred_test), 'F1': f1_score(y_test, y_pred_test),
            'AUC': auc
        })

    ax.plot([0,1],[0,1], linestyle='--', color='gray', label='Random guess')
    ax.set_title(name)
    ax.set_xlabel('False Positive Rate')
    ax.set_ylabel('True Positive Rate')
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig('readmission_window_roc_comparison.png', dpi=150)
plt.show()

windows_df = pd.DataFrame(results_windows)
print(windows_df.to_string(index=False))
windows_df.to_csv('readmission_window_comparison.csv', index=False)

In [ ]:
print(M0['insurance'].value_counts())
print(M0['race'].value_counts())
print(M0['language'].value_counts())
print(M0['marital_status'].value_counts())

In [ ]:
incomplete_mask = (
    (M0['insurance'] == 'Unknown') |
    (M0['race'].isin(['UNKNOWN', 'UNABLE TO OBTAIN', 'PATIENT DECLINED TO ANSWER'])) |
    (M0['language'] == 'Unknown') |
    (M0['marital_status'] == 'Unknown')
)

print("Total patients:", len(M0))
print("Patients with at least one incomplete field:", incomplete_mask.sum())
print("% incomplete:", (incomplete_mask.sum() / len(M0) * 100).round(2))

In [ ]:
M0_complete_only = M0[~incomplete_mask].copy()

print("With incomplete (original):", M0.shape)
print("Without incomplete (filtered):", M0_complete_only.shape)
print("\nTarget distribution — with incomplete:")
print(M0['readmit_30_any'].value_counts(normalize=True))
print("\nTarget distribution — without incomplete:")
print(M0_complete_only['readmit_30_any'].value_counts(normalize=True))

In [ ]:
features_M0 = ['insurance', 'race', 'language', 'marital_status', 'anchor_age']
target = 'readmit_30_any'

X = M0_complete_only[features_M0].copy()
y = M0_complete_only[target]
for col in ['insurance', 'race', 'language', 'marital_status']:
    X[col] = LabelEncoder().fit_transform(X[col].astype(str))

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

rf_M0_complete = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')
rf_M0_complete.fit(X_train, y_train)

y_pred_train = rf_M0_complete.predict(X_train)
y_pred_test = rf_M0_complete.predict(X_test)
y_proba_test = rf_M0_complete.predict_proba(X_test)[:, 1]

print("=== M0 — Without Incomplete Data ===")
print(f"Train Accuracy: {accuracy_score(y_train, y_pred_train):.4f}")
print(f"Test Accuracy:  {accuracy_score(y_test, y_pred_test):.4f}")
print(f"Recall:         {recall_score(y_test, y_pred_test):.4f}")
print(f"Precision:      {precision_score(y_test, y_pred_test):.4f}")
print(f"F1 Score:       {f1_score(y_test, y_pred_test):.4f}")
print(f"AUC:            {roc_auc_score(y_test, y_proba_test):.4f}")

In [ ]:
sdoh_cols = ['Z590', 'Z62810', 'Z560', 'Z5900', 'Z602',
             'homeless', 'alcohol_use', 'drug_use', 'substance_abuse', 'tobacco_use',
             'lives_alone', 'unemployed', 'food_insecure', 'domestic_violence',
             'limited_support', 'financial_stress']

M1_clustered_sdoh = M1_clustered.merge(
    M2_full_5pct[['subject_id', 'hadm_id'] + sdoh_cols],
    on=['subject_id', 'hadm_id'], how='left'
)
M1_clustered_sdoh[sdoh_cols] = M1_clustered_sdoh[sdoh_cols].fillna(0).astype(int)

print(M1_clustered_sdoh.shape)
print(M1_clustered_sdoh['cluster'].value_counts().sort_index())

In [ ]:
baseline_cols = ['insurance', 'race', 'language', 'marital_status', 'anchor_age']
target = 'readmit_30_any'

cluster_results = []

for cluster_id in sorted(M1_clustered_sdoh['cluster'].unique()):
    cluster_df = M1_clustered_sdoh[M1_clustered_sdoh['cluster'] == cluster_id].copy()

    for feature_set_name, features in [('Baseline only', baseline_cols), ('Baseline + SDOH', baseline_cols + sdoh_cols)]:
        X = cluster_df[features].copy()
        y = cluster_df[target]
        for col in ['insurance', 'race', 'language', 'marital_status']:
            X[col] = LabelEncoder().fit_transform(X[col].astype(str))

        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
        rf = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')
        rf.fit(X_train, y_train)
        y_proba_test = rf.predict_proba(X_test)[:, 1]
        auc = roc_auc_score(y_test, y_proba_test)

        cluster_results.append({
            'Cluster': cluster_id, 'Cluster_Size': len(cluster_df),
            'Feature_Set': feature_set_name, 'AUC': auc
        })

        print(f"Cluster {cluster_id} — {feature_set_name}: AUC = {auc:.4f}")

cluster_results_df = pd.DataFrame(cluster_results)
print("\n", cluster_results_df.to_string(index=False))
cluster_results_df.to_csv('cluster_sdoh_benefit.csv', index=False)

In [ ]:
import numpy as np

clusters = [0, 1, 2, 3, 4]
baseline_aucs = [0.5875, 0.6081, 0.5833, 0.5829, 0.6472]
sdoh_aucs = [0.5897, 0.6193, 0.5862, 0.5847, 0.6511]

x = np.arange(len(clusters))
width = 0.35

fig, ax = plt.subplots(figsize=(9, 6))
bars1 = ax.bar(x - width/2, baseline_aucs, width, label='Baseline Only', color='#8B9DC3')
bars2 = ax.bar(x + width/2, sdoh_aucs, width, label='Baseline + SDOH', color='#DC8686')

ax.set_xlabel('Cluster')
ax.set_ylabel('AUC')
ax.set_title('SDOH Benefit by Cluster: Baseline vs. Baseline + SDOH')
ax.set_xticks(x)
ax.set_xticklabels([f'Cluster {c}' for c in clusters])
ax.legend()
ax.set_ylim(0.55, 0.67)

# annotate improvement above each pair
improvements = [s - b for b, s in zip(baseline_aucs, sdoh_aucs)]
for i, imp in enumerate(improvements):
    ax.annotate(f'+{imp:.4f}', xy=(x[i], max(baseline_aucs[i], sdoh_aucs[i]) + 0.003),
                ha='center', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.savefig('cluster_sdoh_benefit_chart.png', dpi=150)
plt.show()

In [ ]:
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler

# Sample 10,000 patients for speed
np.random.seed(42)
sample_idx = np.random.choice(M1_clustered.index, size=10000, replace=False)
M1_sample = M1_clustered.loc[sample_idx].copy()

# Use the same features as clustering (demographics + diagnosis codes, no SDOH)
tsne_cols = ['insurance', 'race', 'language', 'marital_status', 'anchor_age',
             '250', '272', '401', 'E11', 'E78', 'E87', 'I10', 'K21']

X_tsne = M1_sample[tsne_cols].copy()
for col in ['insurance', 'race', 'language', 'marital_status']:
    X_tsne[col] = LabelEncoder().fit_transform(X_tsne[col].astype(str))

X_tsne_scaled = StandardScaler().fit_transform(X_tsne)

print("Running TSNE on", X_tsne_scaled.shape[0], "patients...")
tsne = TSNE(n_components=2, random_state=42, perplexity=30, n_iter=1000)
tsne_result = tsne.fit_transform(X_tsne_scaled)

M1_sample['tsne_x'] = tsne_result[:, 0]
M1_sample['tsne_y'] = tsne_result[:, 1]

print("Done!")
print(M1_sample[['tsne_x', 'tsne_y']].head())

In [ ]:
sdoh_patient_ids = M2_complete_full['subject_id'].unique()
M1_sample['has_sdoh'] = M1_sample['subject_id'].isin(sdoh_patient_ids).astype(int)

plt.figure(figsize=(10, 8))

# plot non-SDOH patients first (background), then SDOH patients on top (foreground)
no_sdoh = M1_sample[M1_sample['has_sdoh'] == 0]
yes_sdoh = M1_sample[M1_sample['has_sdoh'] == 1]

plt.scatter(no_sdoh['tsne_x'], no_sdoh['tsne_y'], s=8, alpha=0.3, color='lightgray', label='No documented SDOH')
plt.scatter(yes_sdoh['tsne_x'], yes_sdoh['tsne_y'], s=15, alpha=0.8, color='crimson', label='Has documented SDOH')

plt.title('TSNE Projection: Patients Colored by Documented SDOH Status')
plt.xlabel('TSNE Dimension 1')
plt.ylabel('TSNE Dimension 2')
plt.legend()
plt.tight_layout()
plt.savefig('tsne_sdoh_projection.png', dpi=150)
plt.show()

print(f"Total patients in sample: {len(M1_sample)}")
print(f"Patients with documented SDOH: {yes_sdoh.shape[0]}")

In [ ]:
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler

# Sample 30,000 patients this time
np.random.seed(42)
sample_idx = np.random.choice(M1_clustered.index, size=30000, replace=False)
M1_sample = M1_clustered.loc[sample_idx].copy()

tsne_cols = ['insurance', 'race', 'language', 'marital_status', 'anchor_age',
             '250', '272', '401', 'E11', 'E78', 'E87', 'I10', 'K21']

X_tsne = M1_sample[tsne_cols].copy()
for col in ['insurance', 'race', 'language', 'marital_status']:
    X_tsne[col] = LabelEncoder().fit_transform(X_tsne[col].astype(str))

X_tsne_scaled = StandardScaler().fit_transform(X_tsne)

print("Running TSNE on", X_tsne_scaled.shape[0], "patients... this may take 5-15 minutes")
tsne = TSNE(n_components=2, random_state=42, perplexity=30, max_iter=1000)
tsne_result = tsne.fit_transform(X_tsne_scaled)

M1_sample['tsne_x'] = tsne_result[:, 0]
M1_sample['tsne_y'] = tsne_result[:, 1]

print("Done!")

In [ ]:
sdoh_patient_ids = M2_complete_full['subject_id'].unique()
M1_sample['has_sdoh'] = M1_sample['subject_id'].isin(sdoh_patient_ids).astype(int)

plt.figure(figsize=(10, 8))
no_sdoh = M1_sample[M1_sample['has_sdoh'] == 0]
yes_sdoh = M1_sample[M1_sample['has_sdoh'] == 1]

plt.scatter(no_sdoh['tsne_x'], no_sdoh['tsne_y'], s=6, alpha=0.25, color='lightgray', label='No documented SDOH')
plt.scatter(yes_sdoh['tsne_x'], yes_sdoh['tsne_y'], s=12, alpha=0.7, color='crimson', label='Has documented SDOH')

plt.title('TSNE Projection (30,000 patients): Colored by Documented SDOH Status')
plt.xlabel('TSNE Dimension 1')
plt.ylabel('TSNE Dimension 2')
plt.legend()
plt.tight_layout()
plt.savefig('tsne_sdoh_30k.png', dpi=150)
plt.show()

print(f"Total patients in sample: {len(M1_sample)}")
print(f"Patients with documented SDOH: {yes_sdoh.shape[0]}")

In [ ]:
plt.figure(figsize=(10, 8))

not_cluster1 = M1_sample[M1_sample['cluster'] != 1]
cluster1 = M1_sample[M1_sample['cluster'] == 1]

plt.scatter(not_cluster1['tsne_x'], not_cluster1['tsne_y'], s=6, alpha=0.25, color='lightgray', label='Other Clusters (0, 2, 3, 4)')
plt.scatter(cluster1['tsne_x'], cluster1['tsne_y'], s=10, alpha=0.6, color='steelblue', label='Cluster 1')

plt.title('TSNE Projection (30,000 patients): Colored by Cluster 1 Membership')
plt.xlabel('TSNE Dimension 1')
plt.ylabel('TSNE Dimension 2')
plt.legend()
plt.tight_layout()
plt.savefig('tsne_cluster1_30k.png', dpi=150)
plt.show()

print(f"Total patients in sample: {len(M1_sample)}")
print(f"Cluster 1 patients in sample: {cluster1.shape[0]}")

In [ ]:
exclude_cols = ['subject_id', 'hadm_id', 'readmit_30_any', 'readmit_30_same_type']
features_M2_complete = [c for c in M2_complete_full.columns if c not in exclude_cols]
target = 'readmit_30_any'

X = M2_complete_full[features_M2_complete].copy()
y = M2_complete_full[target]
for col in ['insurance', 'race', 'language', 'marital_status']:
    X[col] = LabelEncoder().fit_transform(X[col].astype(str))

auc_results = []
acc_results = []
recall_results = []
precision_results = []
f1_results = []

print("Running 10 repeats on M2_complete_full...")
for run in range(10):
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=run, stratify=y
    )
    rf = RandomForestClassifier(n_estimators=100, random_state=run, class_weight='balanced')
    rf.fit(X_train, y_train)
    y_pred_test = rf.predict(X_test)
    y_proba_test = rf.predict_proba(X_test)[:, 1]

    auc = roc_auc_score(y_test, y_proba_test)
    auc_results.append(auc)
    acc_results.append(accuracy_score(y_test, y_pred_test))
    recall_results.append(recall_score(y_test, y_pred_test))
    precision_results.append(precision_score(y_test, y_pred_test))
    f1_results.append(f1_score(y_test, y_pred_test))

    print(f"Run {run+1}: AUC = {auc:.4f}")

print("\n=== Results across 10 runs ===")
for name, results in [('AUC', auc_results), ('Accuracy', acc_results), ('Recall', recall_results),
                       ('Precision', precision_results), ('F1', f1_results)]:
    arr = np.array(results)
    print(f"{name}: mean={arr.mean():.4f}, std={arr.std():.4f}, min={arr.min():.4f}, max={arr.max():.4f}")

ci_results_df = pd.DataFrame({
    'AUC': auc_results, 'Accuracy': acc_results, 'Recall': recall_results,
    'Precision': precision_results, 'F1': f1_results
})
ci_results_df.to_csv('m2_complete_full_10runs_confidence_intervals.csv', index=False)
print("\nSaved m2_complete_full_10runs_confidence_intervals.csv")

In [ ]:
print("=== Full Results: All 10 Runs ===")
print(ci_results_df.round(4).to_string())